# 1.Dataset Merging

## 1.1 merge S2 and U2S2

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import (
    Dataset,
    DataLoader,
    Subset,
    ConcatDataset
)
import numpy as np
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import copy
import torch.nn.functional as F
from sklearn.model_selection import KFold

DATA_DIR = r"data\input"
SPLIT_PATH = os.path.join(DATA_DIR, "Sentinel-2/Landsat-8_splits.json")

save_dir = os.path.join(DATA_DIR, "UAVtoS2+Landsat-8/CSF-PFE")
os.makedirs(save_dir, exist_ok=True)
# =========================
# ⭐ dataset selection
# =========================
selected_datasets = ["sentinel"]

DATASET_CONFIG = {
    "uav": (0, "UAV.csv"),
    "sentinel": (1, "Sentinel-2.csv"),
    "landsat": (2, "Landsat-8.csv"),
}

# =========================
# ⭐ Fusion setting
# =========================
USE_UAV_AUGMENT = True

UAV_SENTINEL_FILE = os.path.join(
    DATA_DIR,
    "UAV_Sentinel_S2A.csv"
)

# =========================
# 0. wavelength normalization
# =========================
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002

def normalize_wavelength(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

def normalize_delta(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

# =========================
# ⭐ split loader
# =========================
def load_split(path, fold_id=0):
    with open(path, "r") as f:
        folds = json.load(f)

    split = folds[fold_id]
    return (
        np.array(split["train_idx"]),
        np.array(split["test_idx"])
    )

# =========================
# Dataset
# =========================
class SpectralDataset(Dataset):
    def __init__(self, file_path, dtype, x_mean, x_std):
        df = pd.read_csv(file_path, encoding='gbk')
        self.dtype = dtype

        temp = df.iloc[:, -2].values.astype(np.float32)
        self.x = df.iloc[:, :-2].values.astype(np.float32)
        self.y = df.iloc[:, -1].values.astype(np.float32)

        # NDWI filter
        if dtype == 1:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 2:
            ndwi = (df['SR_B3'] - df['SR_B5']) / (df['SR_B3'] + df['SR_B5'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # clean
        mask = np.all(self.x >= 0, axis=1)
        self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # global norm
        self.x = (self.x - x_mean) / (x_std + 1e-6)

        # wavelength
        if dtype == 0:
            self.lambda_ = np.arange(350, 1003, 4).astype(np.float32)
            self.delta = np.full_like(self.lambda_, 4.0)
            self.group_id = np.arange(len(self.x)) // 10

        elif dtype == 1:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),(740,15),
                     (783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        else:
            bands = [(443,20),(482,65),(561,75),(655,50),(865,40)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        self.lambda_ = normalize_wavelength(self.lambda_)
        self.delta = normalize_delta(self.delta)

        self.temp = (temp - temp.mean()) / (temp.std() + 1e-6)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(self.x[idx]),
            "lambda": torch.tensor(self.lambda_),
            "delta": torch.tensor(self.delta),
            "y": torch.tensor(self.y[idx]),
            "type": torch.tensor(self.dtype),
            "temp": torch.tensor(self.temp[idx])
        }

# =========================
# Model
# =========================
class SpectralFusionModel(nn.Module):
    def __init__(self, d_model=64, grid_size=8):
        super().__init__()

        self.d_model = d_model

        # =========================
        # 1. Fixed spectral grid
        # =========================
        self.register_buffer(
            "grid",
            torch.linspace(0, 1, grid_size)
        )

        # =========================
        # 2. Fourier Encoding
        # =========================
        self.fourier_B = nn.Parameter(
            torch.logspace(0, 2, d_model // 2)
        )
        self.fourier_scale = nn.Parameter(torch.ones(1))

        # fusion layer to combine spectral and positional embeddings
        self.fusion = nn.Linear(d_model * 2, d_model)

        # =========================
        # 3. Input embedding
        # =========================
        self.input_embed = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.temp_embed = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # =========================
        # 4. learnable spectral weight network
        # =========================
        self.spectral_weight_net = nn.Sequential(
            nn.Linear(4, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, d_model // 4),
            nn.GELU(),
            nn.Linear(d_model // 4, 1),
            nn.Sigmoid()
        )

        # =========================
        # 5. FiLM conditioning
        # =========================
        self.film = nn.Sequential(
            nn.Linear(4, d_model * 2),
            nn.SiLU(),
            nn.Linear(d_model * 2, d_model * 2),
            nn.SiLU(),
            nn.Linear(d_model * 2, d_model * 2)
        )

        # =========================
        # 5. Transformer Encoder
        # =========================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.LayerNorm(d_model)
        )

        # =========================
        # 7. Regression head
        # =========================
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, 1)
        )

    # =========================
    # Fourier Encoding
    # =========================
    def fourier_encoding(self, x):
        x = x.unsqueeze(-1)  # (B, N, 1)

        proj = x * self.fourier_B * self.fourier_scale

        return torch.cat([
            torch.sin(proj),
            torch.cos(proj)
        ], dim=-1)

    # =========================
    # spectral projection
    # =========================
    def spectral_project(self, z, wl, delta):

        grid = self.grid.to(z.device).unsqueeze(0).unsqueeze(0)

        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        sigma = delta + 1e-6

        weight = torch.exp(-((wl - grid) ** 2) / (sigma ** 2))
        weight = weight / (weight.sum(dim=1, keepdim=True) + 1e-8)

        return (z.unsqueeze(2) * weight.unsqueeze(-1)).sum(dim=1)

    # =========================
    # forward
    # =========================
    def forward(self, x, wl, delta, dtype, temp):

        # -------------------------
        # 1. input embedding
        # -------------------------
        x = x.unsqueeze(-1)
        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        inp = torch.cat([x, wl, delta], dim=-1)
        z = self.input_embed(inp)

        # # -------------------------
        # # 2. FiLM conditioning
        # # -------------------------
        # cond = torch.stack([
        #     dtype.float(),
        #     temp,
        #     wl.mean(dim=1).squeeze(-1),
        #     delta.mean(dim=1).squeeze(-1)
        # ], dim=1)

        # gamma, beta = self.film(cond).chunk(2, dim=1)
        # z = gamma.unsqueeze(1) * z + beta.unsqueeze(1)

        # -------------------------
        # 3. spectral projection
        # -------------------------
        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1)
        )

        # -------------------------
        # 4. Fourier positional encoding
        # -------------------------
        grid = self.grid.unsqueeze(0).expand(z.size(0), -1)

        pos = self.fourier_encoding(grid)

        z = torch.cat([z, pos], dim=-1)
        z = self.fusion(z)

        # -------------------------
        # 5. Transformer + CLS token pooling
        # -------------------------
        B = z.size(0)
        t = self.temp_embed(temp.unsqueeze(-1)).unsqueeze(1)
        cls = self.cls_token.expand(B, -1, -1)  # (B,1,d)
        z = torch.cat([t,  z], dim=1)          # (B,N+1,d)

        z = self.encoder(z)

        z = z[:, 0]  # ⭐ CLS token output

        # -------------------------
        # 6. regression head
        # -------------------------
        return self.head(z).squeeze()

# =========================
# Loss
# =========================
def r2_aware_loss(pred, target):
    mse = torch.mean((pred - target) ** 2)

    pred_c = pred - pred.mean()
    tgt_c = target - target.mean()

    corr = torch.sum(pred_c * tgt_c) / (
        torch.sqrt(torch.sum(pred_c ** 2)) *
        torch.sqrt(torch.sum(tgt_c ** 2)) + 1e-8
    )

    return mse + 0.5 * (1 - corr)

# =========================
# ⭐ evaluation
# =========================
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in loader:
            pred = model(
                batch["x"].to(device),
                batch["lambda"].to(device),
                batch["delta"].to(device),
                batch["type"].to(device),
                batch["temp"].to(device)
            )

            y_true.extend(batch["y"].cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mse, rmse, mae, r2


# =========================
# ⭐ train with early stopping
# =========================
def train(EXPERIMENT_MODE="sentinel_test"):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # =========================
    # save dir per experiment
    # =========================
    save_dir = os.path.join(
        DATA_DIR,
        "Sentinel-2/CSF-PE Transformer",
        EXPERIMENT_MODE
    )
    os.makedirs(save_dir, exist_ok=True)

    # =========================
    # dataset paths
    # =========================
    paths = [
        (DATASET_CONFIG[name][0],
         os.path.join(DATA_DIR, DATASET_CONFIG[name][1]))
        for name in selected_datasets
    ]

    # =========================
    # global normalization
    # =========================
    all_x = []

    for _, path in paths:
        df = pd.read_csv(path, encoding='gbk')
        all_x.append(df.iloc[:, :-2].values.astype(np.float32))

    if USE_UAV_AUGMENT:
        uav_df = pd.read_csv(UAV_SENTINEL_FILE, encoding='gbk')
        all_x.append(uav_df.iloc[:, :-2].values.astype(np.float32))

    all_x = np.vstack(all_x)

    x_mean = all_x.mean(axis=0)
    x_std = all_x.std(axis=0) + 1e-6

    # =========================
    # datasets
    # =========================
    dataset = SpectralDataset(
        paths[0][1],
        paths[0][0],
        x_mean,
        x_std
    )

    if USE_UAV_AUGMENT:
        uav_dataset = SpectralDataset(
            UAV_SENTINEL_FILE,
            dtype=1,
            x_mean=x_mean,
            x_std=x_std
        )

        print(f"UAV augmentation samples: {len(uav_dataset)}")

    # =========================
    # 5-fold splits (Sentinel base)
    # =========================
    folds = json.load(open(SPLIT_PATH))

    results = []

    # =========================
    # ⭐ MIXED MODE: prepare full kfold
    # =========================
    if EXPERIMENT_MODE == "mixed":
        full_dataset = ConcatDataset([dataset, uav_dataset])
        full_idx = np.arange(len(full_dataset))
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        mixed_splits = list(kf.split(full_idx))

    # =========================
    # fold loop
    # =========================
    for fold_id in range(5):

        print(f"\n================ FOLD {fold_id} ================")

        # =========================
        # CASE 1: Sentinel test only
        # =========================
        if EXPERIMENT_MODE == "sentinel_test":

            train_idx = np.array(folds[fold_id]["train_idx"])
            test_idx  = np.array(folds[fold_id]["test_idx"])

            sentinel_train_ds = Subset(dataset, train_idx)
            sentinel_test_ds  = Subset(dataset, test_idx)

            if USE_UAV_AUGMENT:
                train_ds = ConcatDataset([
                    sentinel_train_ds,
                    uav_dataset
                ])
            else:
                train_ds = sentinel_train_ds

            test_ds = sentinel_test_ds

        # =========================
        # CASE 2: Mixed CV
        # =========================
        elif EXPERIMENT_MODE == "mixed":

            train_idx, test_idx = mixed_splits[fold_id]

            train_ds = Subset(full_dataset, train_idx)
            test_ds  = Subset(full_dataset, test_idx)

        # =========================
        # print stats
        # =========================
        print(f"Train size: {len(train_ds)}")
        print(f"Test size : {len(test_ds)}")

        # =========================
        # dataloader
        # =========================
        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
        test_loader  = DataLoader(test_ds, batch_size=16)

        # =========================
        # model
        # =========================
        model = SpectralFusionModel().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=3e-4)

        # =========================
        # early stopping
        # =========================
        best_loss = float("inf")
        patience = 50
        wait = 0
        best_model = None

        # =========================
        # training loop
        # =========================
        for epoch in range(500):

            model.train()
            total_loss = 0

            for batch in train_loader:

                pred = model(
                    batch["x"].to(device),
                    batch["lambda"].to(device),
                    batch["delta"].to(device),
                    batch["type"].to(device),
                    batch["temp"].to(device)
                )

                loss = r2_aware_loss(pred, batch["y"].to(device))

                opt.zero_grad()
                loss.backward()
                opt.step()

                total_loss += loss.item()

            # =========================
            # evaluation
            # =========================
            mse, rmse, mae, r2 = evaluate(model, test_loader, device)

            print(
                f"Epoch {epoch:03d} | "
                f"TrainLoss {total_loss:.4f} | "
                f"Test RMSE {rmse:.4f} R2 {r2:.4f}"
            )

            # =========================
            # save best
            # =========================
            if rmse < best_loss:
                best_loss = rmse
                best_model = copy.deepcopy(model.state_dict())
                wait = 0

                # torch.save(
                #     best_model,
                #     os.path.join(
                #         save_dir,
                #         f"best_model_fold{fold_id}.pth"
                #     )
                # )

            else:
                wait += 1
                if wait >= patience:
                    print("Early stopping triggered")
                    break

        # =========================
        # final eval
        # =========================
        model.load_state_dict(best_model)

        mse, rmse, mae, r2 = evaluate(model, test_loader, device)

        print(f"\n[FOLD {fold_id} FINAL]")
        print(f"MSE : {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE : {mae:.4f}")
        print(f"R2  : {r2:.4f}")

        results.append([mse, rmse, mae, r2])

    # =========================
    # summary
    # =========================
    results = np.array(results)

    print("\n================ FINAL 5-FOLD RESULT ================")
    print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
    print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
    print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
    print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

### Scenario A：test data including S2 and U2S2

In [3]:
train(EXPERIMENT_MODE="mixed")

UAV augmentation samples: 570

================ FOLD 0 ================
Train size: 1439
Test size : 360
Epoch 000 | TrainLoss 1860.8750 | Test RMSE 1.9373 R2 -0.2045
Epoch 001 | TrainLoss 271.1670 | Test RMSE 1.5965 R2 0.1820
Epoch 002 | TrainLoss 245.7633 | Test RMSE 1.5546 R2 0.2244
Epoch 003 | TrainLoss 237.8008 | Test RMSE 1.4911 R2 0.2865
Epoch 004 | TrainLoss 233.8430 | Test RMSE 1.4956 R2 0.2822
Epoch 005 | TrainLoss 231.9157 | Test RMSE 1.5254 R2 0.2533
Epoch 006 | TrainLoss 230.0917 | Test RMSE 1.5142 R2 0.2642
Epoch 007 | TrainLoss 226.3084 | Test RMSE 1.4962 R2 0.2815
Epoch 008 | TrainLoss 226.5444 | Test RMSE 1.4977 R2 0.2801
Epoch 009 | TrainLoss 224.7854 | Test RMSE 1.4741 R2 0.3026
Epoch 010 | TrainLoss 222.4094 | Test RMSE 1.4859 R2 0.2915
Epoch 011 | TrainLoss 218.7920 | Test RMSE 1.4854 R2 0.2919
Epoch 012 | TrainLoss 220.8392 | Test RMSE 1.5131 R2 0.2652
Epoch 013 | TrainLoss 222.3865 | Test RMSE 1.4696 R2 0.3069
Epoch 014 | TrainLoss 215.4495 | Test RMSE 1.5011 R2 

### Scenario B：test data including only S2

In [4]:
train(EXPERIMENT_MODE="sentinel_test")

UAV augmentation samples: 570

================ FOLD 0 ================
Train size: 1554
Test size : 245
Epoch 000 | TrainLoss 2282.3669 | Test RMSE 2.0508 R2 -0.4740
Epoch 001 | TrainLoss 299.6937 | Test RMSE 1.5318 R2 0.1777
Epoch 002 | TrainLoss 268.4542 | Test RMSE 1.6048 R2 0.0975
Epoch 003 | TrainLoss 263.4392 | Test RMSE 1.5320 R2 0.1775
Epoch 004 | TrainLoss 255.6284 | Test RMSE 1.5014 R2 0.2100
Epoch 005 | TrainLoss 256.3425 | Test RMSE 1.4815 R2 0.2308
Epoch 006 | TrainLoss 246.6799 | Test RMSE 1.5335 R2 0.1759
Epoch 007 | TrainLoss 242.1494 | Test RMSE 1.4884 R2 0.2237
Epoch 008 | TrainLoss 249.1082 | Test RMSE 1.5207 R2 0.1896
Epoch 009 | TrainLoss 238.8279 | Test RMSE 1.4924 R2 0.2195
Epoch 010 | TrainLoss 233.4107 | Test RMSE 1.4904 R2 0.2215
Epoch 011 | TrainLoss 236.2644 | Test RMSE 1.5357 R2 0.1735
Epoch 012 | TrainLoss 238.5086 | Test RMSE 1.6001 R2 0.1027
Epoch 013 | TrainLoss 231.4126 | Test RMSE 1.5161 R2 0.1945
Epoch 014 | TrainLoss 233.2205 | Test RMSE 1.4801 R2 

## 1.2 merge L8 and U2L8

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import (
    Dataset,
    DataLoader,
    Subset,
    ConcatDataset
)
import numpy as np
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import copy
import torch.nn.functional as F
from sklearn.model_selection import KFold

DATA_DIR = r"data\input"
SPLIT_PATH = os.path.join(DATA_DIR, "Landsat-8/Landsat-8_splits.json")

save_dir = os.path.join(DATA_DIR, "UAVtoL8+Landsat-8/CSF-PFE")
os.makedirs(save_dir, exist_ok=True)
# =========================
# ⭐ dataset selection
# =========================
selected_datasets = ["landsat"]

DATASET_CONFIG = {
    "uav": (0, "UAV.csv"),
    "sentinel": (1, "Sentinel-2.csv"),
    "landsat": (2, "Landsat-8.csv"),
}

# =========================
# ⭐ Fusion setting
# =========================
USE_UAV_AUGMENT = True

UAV_LANDSAT_FILE = os.path.join(
    DATA_DIR,
    "UAV_Landsat8_SRF.csv"
)

# =========================
# 0. wavelength normalization
# =========================
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002

def normalize_wavelength(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

def normalize_delta(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

# =========================
# ⭐ split loader
# =========================
def load_split(path, fold_id=0):
    with open(path, "r") as f:
        folds = json.load(f)

    split = folds[fold_id]
    return (
        np.array(split["train_idx"]),
        np.array(split["test_idx"])
    )

# =========================
# Dataset
# =========================
class SpectralDataset(Dataset):
    def __init__(self, file_path, dtype, x_mean, x_std):
        df = pd.read_csv(file_path, encoding='gbk')
        self.dtype = dtype

        temp = df.iloc[:, -2].values.astype(np.float32)
        self.x = df.iloc[:, :-2].values.astype(np.float32)
        self.y = df.iloc[:, -1].values.astype(np.float32)

        # NDWI filter
        if dtype == 1:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 2:
            ndwi = (df['SR_B3'] - df['SR_B5']) / (df['SR_B3'] + df['SR_B5'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # clean
        mask = np.all(self.x >= 0, axis=1)
        self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # global norm
        self.x = (self.x - x_mean) / (x_std + 1e-6)

        # wavelength
        if dtype == 0:
            self.lambda_ = np.arange(350, 1003, 4).astype(np.float32)
            self.delta = np.full_like(self.lambda_, 4.0)
            self.group_id = np.arange(len(self.x)) // 10

        elif dtype == 1:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),(740,15),
                     (783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        else:
            bands = [(443,20),(482,65),(561,75),(655,50),(865,40)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        self.lambda_ = normalize_wavelength(self.lambda_)
        self.delta = normalize_delta(self.delta)

        self.temp = (temp - temp.mean()) / (temp.std() + 1e-6)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(self.x[idx]),
            "lambda": torch.tensor(self.lambda_),
            "delta": torch.tensor(self.delta),
            "y": torch.tensor(self.y[idx]),
            "type": torch.tensor(self.dtype),
            "temp": torch.tensor(self.temp[idx])
        }

# =========================
# Model
# =========================
class SpectralFusionModel(nn.Module):
    def __init__(self, d_model=64, grid_size=8):
        super().__init__()

        self.d_model = d_model

        # =========================
        # 1. Fixed spectral grid
        # =========================
        self.register_buffer(
            "grid",
            torch.linspace(0, 1, grid_size)
        )

        # =========================
        # 2. Fourier Encoding
        # =========================
        self.fourier_B = nn.Parameter(
            torch.logspace(0, 2, d_model // 2)
        )
        self.fourier_scale = nn.Parameter(torch.ones(1))

        # fusion layer to combine spectral and positional embeddings
        self.fusion = nn.Linear(d_model * 2, d_model)

        # =========================
        # 3. Input embedding
        # =========================
        self.input_embed = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.temp_embed = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # =========================
        # 4. learnable spectral weight network
        # =========================
        self.spectral_weight_net = nn.Sequential(
            nn.Linear(4, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, d_model // 4),
            nn.GELU(),
            nn.Linear(d_model // 4, 1),
            nn.Sigmoid()
        )

        # =========================
        # 5. FiLM conditioning
        # =========================
        self.film = nn.Sequential(
            nn.Linear(4, d_model * 2),
            nn.SiLU(),
            nn.Linear(d_model * 2, d_model * 2),
            nn.SiLU(),
            nn.Linear(d_model * 2, d_model * 2)
        )

        # =========================
        # 5. Transformer Encoder
        # =========================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.LayerNorm(d_model)
        )

        # =========================
        # 7. Regression head
        # =========================
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, 1)
        )

    # =========================
    # Fourier Encoding
    # =========================
    def fourier_encoding(self, x):
        x = x.unsqueeze(-1)  # (B, N, 1)

        proj = x * self.fourier_B * self.fourier_scale

        return torch.cat([
            torch.sin(proj),
            torch.cos(proj)
        ], dim=-1)

    # =========================
    # spectral projection
    # =========================
    def spectral_project(self, z, wl, delta):

        grid = self.grid.to(z.device).unsqueeze(0).unsqueeze(0)

        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        sigma = delta + 1e-6

        weight = torch.exp(-((wl - grid) ** 2) / (sigma ** 2))
        weight = weight / (weight.sum(dim=1, keepdim=True) + 1e-8)

        return (z.unsqueeze(2) * weight.unsqueeze(-1)).sum(dim=1)

    # =========================
    # forward
    # =========================
    def forward(self, x, wl, delta, dtype, temp):

        # -------------------------
        # 1. input embedding
        # -------------------------
        x = x.unsqueeze(-1)
        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        inp = torch.cat([x, wl, delta], dim=-1)
        z = self.input_embed(inp)

        # # -------------------------
        # # 2. FiLM conditioning
        # # -------------------------
        # cond = torch.stack([
        #     dtype.float(),
        #     temp,
        #     wl.mean(dim=1).squeeze(-1),
        #     delta.mean(dim=1).squeeze(-1)
        # ], dim=1)

        # gamma, beta = self.film(cond).chunk(2, dim=1)
        # z = gamma.unsqueeze(1) * z + beta.unsqueeze(1)

        # -------------------------
        # 3. spectral projection
        # -------------------------
        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1)
        )

        # -------------------------
        # 4. Fourier positional encoding
        # -------------------------
        grid = self.grid.unsqueeze(0).expand(z.size(0), -1)

        pos = self.fourier_encoding(grid)

        z = torch.cat([z, pos], dim=-1)
        z = self.fusion(z)

        # -------------------------
        # 5. Transformer + CLS token pooling
        # -------------------------
        B = z.size(0)
        t = self.temp_embed(temp.unsqueeze(-1)).unsqueeze(1)
        cls = self.cls_token.expand(B, -1, -1)  # (B,1,d)
        z = torch.cat([t,  z], dim=1)          # (B,N+1,d)

        z = self.encoder(z)

        z = z[:, 0]  # ⭐ CLS token output

        # -------------------------
        # 6. regression head
        # -------------------------
        return self.head(z).squeeze()

# =========================
# Loss
# =========================
def r2_aware_loss(pred, target):
    mse = torch.mean((pred - target) ** 2)

    pred_c = pred - pred.mean()
    tgt_c = target - target.mean()

    corr = torch.sum(pred_c * tgt_c) / (
        torch.sqrt(torch.sum(pred_c ** 2)) *
        torch.sqrt(torch.sum(tgt_c ** 2)) + 1e-8
    )

    return mse + 0.5 * (1 - corr)

# =========================
# ⭐ evaluation
# =========================
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in loader:
            pred = model(
                batch["x"].to(device),
                batch["lambda"].to(device),
                batch["delta"].to(device),
                batch["type"].to(device),
                batch["temp"].to(device)
            )

            y_true.extend(batch["y"].cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mse, rmse, mae, r2


# =========================
# ⭐ train with early stopping
# =========================
def train(EXPERIMENT_MODE="landsat_test"):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # =========================
    # save dir per experiment
    # =========================
    save_dir = os.path.join(
        DATA_DIR,
        "Landsat-8/CSF-PE Transformer",
        EXPERIMENT_MODE
    )
    os.makedirs(save_dir, exist_ok=True)

    # =========================
    # dataset paths
    # =========================
    paths = [
        (DATASET_CONFIG[name][0],
         os.path.join(DATA_DIR, DATASET_CONFIG[name][1]))
        for name in selected_datasets
    ]

    # =========================
    # global normalization
    # =========================
    all_x = []

    for _, path in paths:
        df = pd.read_csv(path, encoding='gbk')
        all_x.append(df.iloc[:, :-2].values.astype(np.float32))

    if USE_UAV_AUGMENT:
        uav_df = pd.read_csv(UAV_LANDSAT_FILE, encoding='gbk')
        all_x.append(uav_df.iloc[:, :-2].values.astype(np.float32))

    all_x = np.vstack(all_x)

    x_mean = all_x.mean(axis=0)
    x_std = all_x.std(axis=0) + 1e-6

    # =========================
    # datasets
    # =========================
    dataset = SpectralDataset(
        paths[0][1],
        paths[0][0],
        x_mean,
        x_std
    )

    if USE_UAV_AUGMENT:
        uav_dataset = SpectralDataset(
            UAV_LANDSAT_FILE,
            dtype=2,
            x_mean=x_mean,
            x_std=x_std
        )

        print(f"UAV augmentation samples: {len(uav_dataset)}")

    # =========================
    # 5-fold splits (landsat base)
    # =========================
    folds = json.load(open(SPLIT_PATH))

    results = []

    # =========================
    # ⭐ MIXED MODE: prepare full kfold
    # =========================
    if EXPERIMENT_MODE == "mixed":
        full_dataset = ConcatDataset([dataset, uav_dataset])
        full_idx = np.arange(len(full_dataset))
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        mixed_splits = list(kf.split(full_idx))

    # =========================
    # fold loop
    # =========================
    for fold_id in range(5):

        print(f"\n================ FOLD {fold_id} ================")

        # =========================
        # CASE 1: landsat test only
        # =========================
        if EXPERIMENT_MODE == "landsat_test":

            train_idx = np.array(folds[fold_id]["train_idx"])
            test_idx  = np.array(folds[fold_id]["test_idx"])

            landsat_train_ds = Subset(dataset, train_idx)
            landsat_test_ds  = Subset(dataset, test_idx)

            if USE_UAV_AUGMENT:
                train_ds = ConcatDataset([
                    landsat_train_ds,
                    uav_dataset
                ])
            else:
                train_ds = landsat_train_ds

            test_ds = landsat_test_ds

        # =========================
        # CASE 2: Mixed CV
        # =========================
        elif EXPERIMENT_MODE == "mixed":

            train_idx, test_idx = mixed_splits[fold_id]

            train_ds = Subset(full_dataset, train_idx)
            test_ds  = Subset(full_dataset, test_idx)

        # =========================
        # print stats
        # =========================
        print(f"Train size: {len(train_ds)}")
        print(f"Test size : {len(test_ds)}")

        # =========================
        # dataloader
        # =========================
        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
        test_loader  = DataLoader(test_ds, batch_size=16)

        # =========================
        # model
        # =========================
        model = SpectralFusionModel().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=3e-4)

        # =========================
        # early stopping
        # =========================
        best_loss = float("inf")
        patience = 50
        wait = 0
        best_model = None

        # =========================
        # training loop
        # =========================
        for epoch in range(500):

            model.train()
            total_loss = 0

            for batch in train_loader:

                pred = model(
                    batch["x"].to(device),
                    batch["lambda"].to(device),
                    batch["delta"].to(device),
                    batch["type"].to(device),
                    batch["temp"].to(device)
                )

                loss = r2_aware_loss(pred, batch["y"].to(device))

                opt.zero_grad()
                loss.backward()
                opt.step()

                total_loss += loss.item()

            # =========================
            # evaluation
            # =========================
            mse, rmse, mae, r2 = evaluate(model, test_loader, device)

            print(
                f"Epoch {epoch:03d} | "
                f"TrainLoss {total_loss:.4f} | "
                f"Test RMSE {rmse:.4f} R2 {r2:.4f}"
            )

            # =========================
            # save best
            # =========================
            if rmse < best_loss:
                best_loss = rmse
                best_model = copy.deepcopy(model.state_dict())
                wait = 0

                # torch.save(
                #     best_model,
                #     os.path.join(
                #         save_dir,
                #         f"best_model_fold{fold_id}.pth"
                #     )
                # )

            else:
                wait += 1
                if wait >= patience:
                    print("Early stopping triggered")
                    break

        # =========================
        # final eval
        # =========================
        model.load_state_dict(best_model)

        mse, rmse, mae, r2 = evaluate(model, test_loader, device)

        print(f"\n[FOLD {fold_id} FINAL]")
        print(f"MSE : {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE : {mae:.4f}")
        print(f"R2  : {r2:.4f}")

        results.append([mse, rmse, mae, r2])

    # =========================
    # summary
    # =========================
    results = np.array(results)

    print("\n================ FINAL 5-FOLD RESULT ================")
    print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
    print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
    print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
    print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

### Scenario A：test data including L8 and U2L8

In [12]:
train(EXPERIMENT_MODE="mixed")

UAV augmentation samples: 570

================ FOLD 0 ================
Train size: 968
Test size : 242
Epoch 000 | TrainLoss 1770.2622 | Test RMSE 3.3726 R2 -2.3478
Epoch 001 | TrainLoss 401.8600 | Test RMSE 1.5480 R2 0.2947
Epoch 002 | TrainLoss 160.3726 | Test RMSE 1.7086 R2 0.1407
Epoch 003 | TrainLoss 156.7079 | Test RMSE 1.5047 R2 0.3336
Epoch 004 | TrainLoss 154.9323 | Test RMSE 1.6342 R2 0.2139
Epoch 005 | TrainLoss 152.6900 | Test RMSE 1.6200 R2 0.2276
Epoch 006 | TrainLoss 147.6407 | Test RMSE 1.4812 R2 0.3542
Epoch 007 | TrainLoss 145.0819 | Test RMSE 1.4868 R2 0.3493
Epoch 008 | TrainLoss 146.4232 | Test RMSE 1.5695 R2 0.2750
Epoch 009 | TrainLoss 139.0304 | Test RMSE 1.5080 R2 0.3306
Epoch 010 | TrainLoss 142.5345 | Test RMSE 1.5130 R2 0.3262
Epoch 011 | TrainLoss 141.8778 | Test RMSE 1.5464 R2 0.2961
Epoch 012 | TrainLoss 139.0806 | Test RMSE 1.5515 R2 0.2915
Epoch 013 | TrainLoss 135.5010 | Test RMSE 1.5015 R2 0.3364
Epoch 014 | TrainLoss 138.3843 | Test RMSE 1.4862 R2 0

### Scenario B：test data including only L8

In [13]:
train(EXPERIMENT_MODE="landsat_test")

UAV augmentation samples: 570

================ FOLD 0 ================
Train size: 1082
Test size : 128
Epoch 000 | TrainLoss 2069.5387 | Test RMSE 3.8590 R2 -3.8295
Epoch 001 | TrainLoss 508.6476 | Test RMSE 1.7811 R2 -0.0288
Epoch 002 | TrainLoss 177.7740 | Test RMSE 1.6974 R2 0.0656
Epoch 003 | TrainLoss 164.1022 | Test RMSE 1.7431 R2 0.0146
Epoch 004 | TrainLoss 164.4026 | Test RMSE 1.6439 R2 0.1236
Epoch 005 | TrainLoss 159.4162 | Test RMSE 1.6539 R2 0.1129
Epoch 006 | TrainLoss 156.7778 | Test RMSE 1.6086 R2 0.1609
Epoch 007 | TrainLoss 154.2989 | Test RMSE 1.6195 R2 0.1494
Epoch 008 | TrainLoss 151.0645 | Test RMSE 1.5525 R2 0.2183
Epoch 009 | TrainLoss 150.8570 | Test RMSE 1.5771 R2 0.1934
Epoch 010 | TrainLoss 151.8841 | Test RMSE 1.6095 R2 0.1599
Epoch 011 | TrainLoss 150.4987 | Test RMSE 1.5708 R2 0.1998
Epoch 012 | TrainLoss 147.5476 | Test RMSE 1.6100 R2 0.1593
Epoch 013 | TrainLoss 144.2230 | Test RMSE 1.5987 R2 0.1711
Epoch 014 | TrainLoss 150.0932 | Test RMSE 1.6491 R2

# 2.Pretrain the model and freeze fine-tuning

## UAV pretrain transfer to Sentinel 2

### UAV2S2A_transfer

#### UAV2S2A_model_acc_test

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import copy
import torch.nn.functional as F

DATA_DIR = r"E:\LYQ\work\250916remote_model\data\input"
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")
model_save_dir = os.path.join(DATA_DIR, "Sentinel2_transfer/UAV2S2A/PI-Transformer")
os.makedirs(model_save_dir, exist_ok=True)

# =========================
# ⭐ dataset selection
# =========================
selected_datasets = ["uav2s2a"]

DATASET_CONFIG = {
    "uav": (0, "UAV.csv"),
    "sentinel": (1, "Sentinel-2.csv"),
    "landsat": (2, "Landsat-8.csv"),
    "uav2s2a": (3, "UAV2S2A.csv")

}

# =========================
# 0. wavelength normalization
# =========================
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002

def normalize_wavelength(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

def normalize_delta(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

# =========================
# ⭐ split loader
# =========================
def load_split(path, fold_id=0):
    with open(path, "r") as f:
        folds = json.load(f)

    split = folds[fold_id]
    return (
        np.array(split["train_idx"]),
        np.array(split["test_idx"])
    )

# =========================
# Dataset
# =========================
class SpectralDataset(Dataset):
    def __init__(self, file_path, dtype, x_mean, x_std):
        df = pd.read_csv(file_path, encoding='gbk')
        self.dtype = dtype

        temp = df.iloc[:, -2].values.astype(np.float32)
        self.x = df.iloc[:, :-2].values.astype(np.float32)
        self.y = df.iloc[:, -1].values.astype(np.float32)

        # NDWI filter
        if dtype == 1:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 2:
            ndwi = (df['SR_B3'] - df['SR_B5']) / (df['SR_B3'] + df['SR_B5'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # clean
        mask = np.all(self.x >= 0, axis=1)
        self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # global norm
        self.x = (self.x - x_mean) / (x_std + 1e-6)

        # wavelength
        if dtype == 0:
            self.lambda_ = np.arange(350, 1003, 4).astype(np.float32)
            self.delta = np.full_like(self.lambda_, 4.0)
            self.group_id = np.arange(len(self.x)) // 10

        elif dtype == 1 or dtype == 3:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),
                     (740,15),(783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        elif dtype == 2:
            bands = [(443,20),(482,65),(561,75),(655,50),(865,40)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        self.lambda_ = normalize_wavelength(self.lambda_)
        self.delta = normalize_delta(self.delta)

        self.temp = (temp - temp.mean()) / (temp.std() + 1e-6)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(self.x[idx]),
            "lambda": torch.tensor(self.lambda_),
            "delta": torch.tensor(self.delta),
            "y": torch.tensor(self.y[idx]),
            "type": torch.tensor(self.dtype),
            "temp": torch.tensor(self.temp[idx])
        }

# =========================
# Model
# =========================
class SpectralFusionModel(nn.Module):
    def __init__(self, d_model=16, grid_size=16):
        super().__init__()

        self.d_model = d_model

        # =========================
        # 1. Fixed spectral grid
        # =========================
        self.register_buffer(
            "grid",
            torch.linspace(0, 1, grid_size)
        )

        # =========================
        # 2. Fourier Encoding
        # =========================
        self.fourier_B = nn.Parameter(
            torch.logspace(0, 2, d_model // 2)
        )
        self.fourier_scale = nn.Parameter(torch.ones(1))

        # fusion layer to combine spectral and positional embeddings
        self.fusion = nn.Linear(d_model * 2, d_model)

        # =========================
        # 3. Input embedding
        # =========================
        self.input_embed = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.temp_embed = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # =========================
        # 4. FiLM conditioning
        # =========================
        self.film = nn.Sequential(
            nn.Linear(4, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model * 2)
        )

        # =========================
        # 5. Transformer Encoder
        # =========================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # =========================
        # 7. Regression head
        # =========================
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.ReLU(),
            nn.Linear(d_model//2, 1)
        )

    # =========================
    # Fourier Encoding
    # =========================
    def fourier_encoding(self, x):
        x = x.unsqueeze(-1)  # (B, N, 1)

        proj = x * self.fourier_B * self.fourier_scale

        return torch.cat([
            torch.sin(proj),
            torch.cos(proj)
        ], dim=-1)

    # =========================
    # spectral projection
    # =========================
    def spectral_project(self, z, wl, delta):

        grid = self.grid.to(z.device).unsqueeze(0).unsqueeze(0)

        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        sigma = delta + 1e-6

        weight = torch.exp(-((wl - grid) ** 2) / (sigma ** 2))
        weight = weight / (weight.sum(dim=1, keepdim=True) + 1e-8)

        return (z.unsqueeze(2) * weight.unsqueeze(-1)).sum(dim=1)

    # =========================
    # forward
    # =========================
    def forward(self, x, wl, delta, dtype, temp):

        # -------------------------
        # 1. input embedding
        # -------------------------
        x = x.unsqueeze(-1)
        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        inp = torch.cat([x, wl, delta], dim=-1)
        z = self.input_embed(inp)

        # -------------------------
        # 2. FiLM conditioning
        # -------------------------
        cond = torch.stack([
            dtype.float(),
            temp,
            wl.mean(dim=1).squeeze(-1),
            delta.mean(dim=1).squeeze(-1)
        ], dim=1)

        gamma, beta = self.film(cond).chunk(2, dim=1)
        z = gamma.unsqueeze(1) * z + beta.unsqueeze(1)

        # -------------------------
        # 3. spectral projection
        # -------------------------
        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1)
        )

        # -------------------------
        # 4. Fourier positional encoding
        # -------------------------
        grid = self.grid.unsqueeze(0).expand(z.size(0), -1)

        pos = self.fourier_encoding(grid)

        z = torch.cat([z, pos], dim=-1)
        z = self.fusion(z)

        # -------------------------
        # 5. Transformer + CLS token pooling
        # -------------------------
        B = z.size(0)
        t = self.temp_embed(temp.unsqueeze(-1)).unsqueeze(1)
        z = torch.cat([t, z], dim=1)          # (B,N+1,d)

        z = self.encoder(z)

        z = z[:, 0]  # ⭐ CLS token output

        # -------------------------
        # 6. regression head
        # -------------------------
        return self.head(z).squeeze()

# =========================
# Loss
# =========================
def r2_aware_loss(pred, target):
    mse = torch.mean((pred - target) ** 2)

    pred_c = pred - pred.mean()
    tgt_c = target - target.mean()

    corr = torch.sum(pred_c * tgt_c) / (
        torch.sqrt(torch.sum(pred_c ** 2)) *
        torch.sqrt(torch.sum(tgt_c ** 2)) + 1e-8
    )

    return mse + 0.5 * (1 - corr)

# =========================
# ⭐ evaluation
# =========================
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in loader:
            pred = model(
                batch["x"].to(device),
                batch["lambda"].to(device),
                batch["delta"].to(device),
                batch["type"].to(device),
                batch["temp"].to(device)
            )

            y_true.extend(batch["y"].cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mse, rmse, mae, r2


# =========================
# ⭐ train with early stopping
# =========================
def train():

    device = "cuda" if torch.cuda.is_available() else "cpu"

    paths = [
        (DATASET_CONFIG[name][0],
         os.path.join(DATA_DIR, DATASET_CONFIG[name][1]))
        for name in selected_datasets
    ]

    # =========================
    # global normalization
    # =========================
    all_x = []
    for _, path in paths:
        df = pd.read_csv(path, encoding='gbk')
        all_x.append(df.iloc[:, :-2].values.astype(np.float32))

    all_x = np.vstack(all_x)
    x_mean = all_x.mean(axis=0)
    x_std = all_x.std(axis=0) + 1e-6

    dataset = SpectralDataset(paths[0][1], paths[0][0], x_mean, x_std)

    # =========================
    # ⭐ 5-fold split
    # =========================
    folds = json.load(open(SPLIT_PATH))

    results = []

    for fold_id in range(5):

        print(f"\n================ FOLD {fold_id} ================")

        train_idx = np.array(folds[fold_id]["train_idx"])
        test_idx  = np.array(folds[fold_id]["test_idx"])

        train_ds = Subset(dataset, train_idx)
        test_ds  = Subset(dataset, test_idx)

        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=16)

        model = SpectralFusionModel().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=3e-4)

        # =========================
        # ⭐ early stopping
        # =========================
        best_loss = float("inf")
        patience = 50
        wait = 0
        best_model = None

        for epoch in range(500):

            model.train()
            total_loss = 0

            for batch in train_loader:
                pred = model(
                    batch["x"].to(device),
                    batch["lambda"].to(device),
                    batch["delta"].to(device),
                    batch["type"].to(device),
                    batch["temp"].to(device)
                )

                loss = r2_aware_loss(pred, batch["y"].to(device))

                opt.zero_grad()
                loss.backward()
                opt.step()

                total_loss += loss.item()

            # =========================
            # ⭐ validation = test here
            # =========================
            mse, rmse, mae, r2 = evaluate(model, test_loader, device)

            print(
                f"Epoch {epoch:03d} | "
                f"TrainLoss {total_loss:.4f} | "
                f"Test RMSE {rmse:.4f} R2 {r2:.4f}"
            )

            # =========================
            # ⭐ save best model
            # =========================
            if rmse < best_loss:
                best_loss = rmse
                best_model = copy.deepcopy(model.state_dict())
                wait = 0

                torch.save(
                    best_model,
                    os.path.join(model_save_dir, f"best_model_fold{fold_id}.pth")
                )
            else:
                wait += 1
                if wait >= patience:
                    print("Early stopping triggered")
                    break

        # =========================
        # ⭐ final evaluation
        # =========================
        model.load_state_dict(best_model)

        mse, rmse, mae, r2 = evaluate(model, test_loader, device)

        print(f"\n[FOLD {fold_id} FINAL]")
        print(f"MSE: {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE: {mae:.4f}")
        print(f"R2: {r2:.4f}")

        results.append([mse, rmse, mae, r2])

    # =========================
    # ⭐ overall result
    # =========================
    results = np.array(results)

    print("\n================ FINAL 5-FOLD RESULT ================")
    print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
    print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
    print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
    print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

if __name__ == "__main__":
    train()


================ FOLD 0 ================
Epoch 000 | TrainLoss 1691.5449 | Test RMSE 7.3367 R2 -19.6676
Epoch 001 | TrainLoss 1647.1105 | Test RMSE 7.2581 R2 -19.2270
Epoch 002 | TrainLoss 1612.2188 | Test RMSE 7.1760 R2 -18.7722
Epoch 003 | TrainLoss 1573.6394 | Test RMSE 7.0844 R2 -18.2709
Epoch 004 | TrainLoss 1535.3835 | Test RMSE 6.9852 R2 -17.7348
Epoch 005 | TrainLoss 1493.8612 | Test RMSE 6.8833 R2 -17.1920
Epoch 006 | TrainLoss 1449.5405 | Test RMSE 6.7726 R2 -16.6118
Epoch 007 | TrainLoss 1400.4339 | Test RMSE 6.6474 R2 -15.9666
Epoch 008 | TrainLoss 1351.0784 | Test RMSE 6.5142 R2 -15.2935
Epoch 009 | TrainLoss 1292.0883 | Test RMSE 6.3789 R2 -14.6238
Epoch 010 | TrainLoss 1236.4627 | Test RMSE 6.2260 R2 -13.8836
Epoch 011 | TrainLoss 1180.7621 | Test RMSE 6.0669 R2 -13.1328
Epoch 012 | TrainLoss 1123.5059 | Test RMSE 5.9018 R2 -12.3739
Epoch 013 | TrainLoss 1059.8902 | Test RMSE 5.7205 R2 -11.5650
Epoch 014 | TrainLoss 997.7746 | Test RMSE 5.5345 R2 -10.7610
Epoch 015 | Tr

#### UAV2S2A_model_train

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import copy
import torch.nn.functional as F

DATA_DIR = r"\data\input"
model_save_dir = os.path.join(DATA_DIR, "UAV2S2A/train_full/PI-Transformer")
os.makedirs(model_save_dir, exist_ok=True)

# =========================
# ⭐ dataset selection
# =========================
selected_datasets = ["uav2s2a"]

DATASET_CONFIG = {
    "uav": (0, "UAV.csv"),
    "sentinel": (1, "Sentinel-2.csv"),
    "landsat": (2, "Landsat-8.csv"),
    "uav2s2a": (3, "UAV2S2A.csv")

}

# =========================
# 0. wavelength normalization
# =========================
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002

def normalize_wavelength(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

def normalize_delta(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

# =========================
# Dataset
# =========================
class SpectralDataset(Dataset):
    def __init__(self, file_path, dtype, x_mean, x_std):
        df = pd.read_csv(file_path, encoding='gbk')
        self.dtype = dtype

        temp = df.iloc[:, -2].values.astype(np.float32)
        self.x = df.iloc[:, :-2].values.astype(np.float32)
        self.y = df.iloc[:, -1].values.astype(np.float32)

        # NDWI filter
        if dtype == 1:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 2:
            ndwi = (df['SR_B3'] - df['SR_B5']) / (df['SR_B3'] + df['SR_B5'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 3:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # clean
        mask = np.all(self.x >= 0, axis=1)
        self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # global norm
        self.x = (self.x - x_mean) / (x_std + 1e-6)

        # wavelength
        if dtype == 0:
            self.lambda_ = np.arange(350, 1003, 4).astype(np.float32)
            self.delta = np.full_like(self.lambda_, 4.0)
            self.group_id = np.arange(len(self.x)) // 10

        elif dtype == 1:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),
                     (740,15),(783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        elif dtype == 2:
            bands = [(443,20),(482,65),(561,75),(655,50),(865,40)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        elif dtype == 3:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),
                     (740,15),(783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        self.lambda_ = normalize_wavelength(self.lambda_)
        self.delta = normalize_delta(self.delta)

        self.temp = (temp - temp.mean()) / (temp.std() + 1e-6)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(self.x[idx]),
            "lambda": torch.tensor(self.lambda_),
            "delta": torch.tensor(self.delta),
            "y": torch.tensor(self.y[idx]),
            "type": torch.tensor(self.dtype),
            "temp": torch.tensor(self.temp[idx])
        }

# =========================
# Model
# =========================
class SpectralFusionModel(nn.Module):
    def __init__(self, d_model=16, grid_size=16):
        super().__init__()

        self.d_model = d_model

        # =========================
        # 1. Fixed spectral grid
        # =========================
        self.register_buffer(
            "grid",
            torch.linspace(0, 1, grid_size)
        )

        # =========================
        # 2. Fourier Encoding
        # =========================
        self.fourier_B = nn.Parameter(
            torch.logspace(0, 2, d_model // 2)
        )
        self.fourier_scale = nn.Parameter(torch.ones(1))

        # fusion layer to combine spectral and positional embeddings
        self.fusion = nn.Linear(d_model * 2, d_model)

        # =========================
        # 3. Input embedding
        # =========================
        self.input_embed = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.temp_embed = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # =========================
        # 4. FiLM conditioning
        # =========================
        self.film = nn.Sequential(
            nn.Linear(4, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model * 2)
        )

        # =========================
        # 5. Transformer Encoder
        # =========================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # =========================
        # 7. Regression head
        # =========================
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.ReLU(),
            nn.Linear(d_model//2, 1)
        )

    # =========================
    # Fourier Encoding
    # =========================
    def fourier_encoding(self, x):
        x = x.unsqueeze(-1)  # (B, N, 1)

        proj = x * self.fourier_B * self.fourier_scale

        return torch.cat([
            torch.sin(proj),
            torch.cos(proj)
        ], dim=-1)

    # =========================
    # spectral projection
    # =========================
    def spectral_project(self, z, wl, delta):

        grid = self.grid.to(z.device).unsqueeze(0).unsqueeze(0)

        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        sigma = delta + 1e-6

        weight = torch.exp(-((wl - grid) ** 2) / (sigma ** 2))
        weight = weight / (weight.sum(dim=1, keepdim=True) + 1e-8)

        return (z.unsqueeze(2) * weight.unsqueeze(-1)).sum(dim=1)

    # =========================
    # forward
    # =========================
    def forward(self, x, wl, delta, dtype, temp):

        # -------------------------
        # 1. input embedding
        # -------------------------
        x = x.unsqueeze(-1)
        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        inp = torch.cat([x, wl, delta], dim=-1)
        z = self.input_embed(inp)

        # -------------------------
        # 2. FiLM conditioning
        # -------------------------
        cond = torch.stack([
            dtype.float(),
            temp,
            wl.mean(dim=1).squeeze(-1),
            delta.mean(dim=1).squeeze(-1)
        ], dim=1)

        gamma, beta = self.film(cond).chunk(2, dim=1)
        z = gamma.unsqueeze(1) * z + beta.unsqueeze(1)

        # -------------------------
        # 3. spectral projection
        # -------------------------
        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1)
        )

        # -------------------------
        # 4. Fourier positional encoding
        # -------------------------
        grid = self.grid.unsqueeze(0).expand(z.size(0), -1)

        pos = self.fourier_encoding(grid)

        z = torch.cat([z, pos], dim=-1)
        z = self.fusion(z)

        # -------------------------
        # 5. Transformer + CLS token pooling
        # -------------------------
        B = z.size(0)
        t = self.temp_embed(temp.unsqueeze(-1)).unsqueeze(1)
        z = torch.cat([t, z], dim=1)          # (B,N+1,d)

        z = self.encoder(z)

        z = z[:, 0]  # ⭐ CLS token output

        # -------------------------
        # 6. regression head
        # -------------------------
        return self.head(z).squeeze()

# =========================
# Loss
# =========================
def r2_aware_loss(pred, target):
    mse = torch.mean((pred - target) ** 2)

    pred_c = pred - pred.mean()
    tgt_c = target - target.mean()

    corr = torch.sum(pred_c * tgt_c) / (
        torch.sqrt(torch.sum(pred_c ** 2)) *
        torch.sqrt(torch.sum(tgt_c ** 2)) + 1e-8
    )

    return mse + 0.5 * (1 - corr)

# =========================
# ⭐ train on full dataset
# =========================
def train():

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # =========================
    # save dir
    # =========================
    save_dir = os.path.join(
        DATA_DIR,
        "UAV2S2A/train_full/PI-Transformer"
    )
    os.makedirs(save_dir, exist_ok=True)

    paths = [
        (DATASET_CONFIG[name][0],
         os.path.join(DATA_DIR, DATASET_CONFIG[name][1]))
        for name in selected_datasets
    ]

    # =========================
    # global normalization
    # =========================
    all_x = []

    for _, path in paths:
        df = pd.read_csv(path, encoding='gbk')
        all_x.append(df.iloc[:, :-2].values.astype(np.float32))

    all_x = np.vstack(all_x)

    x_mean = all_x.mean(axis=0)
    x_std = all_x.std(axis=0) + 1e-6

    # =========================
    # full dataset
    # =========================
    dataset = SpectralDataset(
        paths[0][1],
        paths[0][0],
        x_mean,
        x_std
    )

    train_loader = DataLoader(
        dataset,
        batch_size=16,
        shuffle=True
    )

    # =========================
    # model
    # =========================
    model = SpectralFusionModel().to(device)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=3e-4
    )

    # =========================
    # train
    # =========================
    epochs = 150

    for epoch in range(epochs):

        model.train()

        total_loss = 0

        for batch in train_loader:

            pred = model(
                batch["x"].to(device),
                batch["lambda"].to(device),
                batch["delta"].to(device),
                batch["type"].to(device),
                batch["temp"].to(device)
            )

            loss = r2_aware_loss(
                pred,
                batch["y"].to(device)
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()

        # =========================
        # evaluate on training set
        # =========================
        mse, rmse, mae, r2 = evaluate(
            model,
            train_loader,
            device
        )

        print(
            f"Epoch {epoch:03d} | "
            f"TrainLoss {total_loss:.4f} | "
            f"Train RMSE {rmse:.4f} | "
            f"Train MAE {mae:.4f} | "
            f"Train R2 {r2:.4f}"
        )

    # =========================
    # save final model
    # =========================
    model_path = os.path.join(
        save_dir,
        "PI_Transformer_full.pth"
    )

    torch.save(model.state_dict(), model_path)

    print("\nTraining Finished")
    print(f"Model saved to:\n{model_path}")

if __name__ == "__main__":
    train()

Epoch 000 | TrainLoss 2011.2715 | Train RMSE 7.3799 | Train MAE 7.1476 | Train R2 -15.1365
Epoch 001 | TrainLoss 1968.7272 | Train RMSE 7.3250 | Train MAE 7.0909 | Train R2 -14.8975
Epoch 002 | TrainLoss 1932.7461 | Train RMSE 7.1281 | Train MAE 6.8872 | Train R2 -14.0540
Epoch 003 | TrainLoss 1812.2651 | Train RMSE 6.9184 | Train MAE 6.6700 | Train R2 -13.1813
Epoch 004 | TrainLoss 1699.0654 | Train RMSE 6.6894 | Train MAE 6.4322 | Train R2 -12.2584
Epoch 005 | TrainLoss 1582.0126 | Train RMSE 6.4317 | Train MAE 6.1637 | Train R2 -11.2564
Epoch 006 | TrainLoss 1458.5304 | Train RMSE 6.1496 | Train MAE 5.8688 | Train R2 -10.2048
Epoch 007 | TrainLoss 1323.8043 | Train RMSE 5.8266 | Train MAE 5.5294 | Train R2 -9.0587
Epoch 008 | TrainLoss 1176.7718 | Train RMSE 5.4786 | Train MAE 5.1615 | Train R2 -7.8931
Epoch 009 | TrainLoss 1037.7916 | Train RMSE 5.0958 | Train MAE 4.7538 | Train R2 -6.6937
Epoch 010 | TrainLoss 887.8015 | Train RMSE 4.7075 | Train MAE 4.3500 | Train R2 -5.5660
Epoc

#### transfer

In [ ]:
import os
import copy
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from torch.utils.data import (
    Dataset,
    DataLoader,
    random_split
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# =========================================================
# PATH
# =========================================================

DATA_DIR = r"\data\input"
SPLIT_PATH = os.path.join(DATA_DIR, "Sentinel-2/Sentinel-2_splits.json")

S2_PATH = os.path.join(
    DATA_DIR,
    "Sentinel-2.csv"
)

# =========================================================
# UAV -> S2A pretrained model
# =========================================================

PRETRAIN_PATH = (
    r"\data\input"
    r"\UAV2S2A\train_full\PI-Transformer"
    r"\PI_Transformer_full.pth"
)

SAVE_DIR = os.path.join(
    DATA_DIR,
    "Sentinel2_transfer"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# =========================================================
# GLOBAL NORMALIZATION
# =========================================================

GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002.0

def normalize_wavelength(x):

    return (
        (x - GLOBAL_LAMBDA_MIN)
        / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)
    )

# =========================================================
# DATASET
# =========================================================

class SentinelDataset(Dataset):

    def __init__(self, csv_path):

        df = pd.read_csv(
            csv_path,
            encoding="gbk"
        )

        # -------------------------------------------------
        # Sentinel-2 bands
        # -------------------------------------------------

        self.band_names = [
            "B1",
            "B2",
            "B3",
            "B4",
            "B5",
            "B6",
            "B7",
            "B8",
            "B8A",
            "B9"
        ]

        # -------------------------------------------------
        # center wavelength
        # -------------------------------------------------

        self.lambda_ = np.array([
            443,
            490,
            560,
            665,
            705,
            740,
            783,
            842,
            865,
            945

        ], dtype=np.float32)

        # -------------------------------------------------
        # bandwidth
        # -------------------------------------------------

        self.delta = np.array([
            20,
            65,
            35,
            30,
            15,
            15,
            20,
            105,
            20,
            20

        ], dtype=np.float32)

        # -------------------------------------------------
        # spectral input
        # -------------------------------------------------

        self.x = df[
            self.band_names
        ].values.astype(np.float32)

        # -------------------------------------------------
        # target
        # -------------------------------------------------

        self.y = (
            df.iloc[:, -1]
            .values
            .astype(np.float32)
        )

        # -------------------------------------------------
        # water temperature
        # -------------------------------------------------

        temp = (
            df.iloc[:, -2]
            .values
            .astype(np.float32)
        )

        # -------------------------------------------------
        # NDWI filter
        # -------------------------------------------------

        ndwi = (
            df["B3"] - df["B8"]
        ) / (
            df["B3"] + df["B8"] + 1e-8
        )

        mask = ndwi > 0.1

        self.x = self.x[mask]

        self.y = self.y[mask]

        temp = temp[mask]

        # -------------------------------------------------
        # remove invalid
        # -------------------------------------------------

        mask = np.all(
            self.x >= 0,
            axis=1
        )

        self.x = self.x[mask]

        self.y = self.y[mask]

        temp = temp[mask]

        # -------------------------------------------------
        # normalize spectral
        # -------------------------------------------------

        self.x_mean = self.x.mean(axis=0)

        self.x_std = (
            self.x.std(axis=0)
            + 1e-6
        )

        self.x = (
            self.x - self.x_mean
        ) / self.x_std

        # -------------------------------------------------
        # normalize temperature
        # -------------------------------------------------

        self.temp = (
            temp - temp.mean()
        ) / (
            temp.std() + 1e-6
        )

        # -------------------------------------------------
        # normalize wavelength
        # -------------------------------------------------

        self.lambda_ = normalize_wavelength(
            self.lambda_
        )

        self.delta = normalize_wavelength(
            self.delta
        )

    def __len__(self):

        return len(self.x)

    def __getitem__(self, idx):

        return {

            "x": torch.tensor(
                self.x[idx]
            ),

            "lambda": torch.tensor(
                self.lambda_
            ),

            "delta": torch.tensor(
                self.delta
            ),

            "temp": torch.tensor(
                self.temp[idx]
            ),

            # Sentinel type = 1
            "type": torch.tensor(1),

            "y": torch.tensor(
                self.y[idx]
            )
        }

# =========================================================
# MODEL
# =========================================================

class SpectralFusionModel(nn.Module):

    def __init__(
        self,
        d_model=16,
        grid_size=16
    ):

        super().__init__()

        self.d_model = d_model

        # =================================================
        # spectral grid
        # =================================================

        self.register_buffer(

            "grid",

            torch.linspace(
                0,
                1,
                grid_size
            )
        )

        # =================================================
        # Fourier encoding
        # =================================================

        self.fourier_B = nn.Parameter(

            torch.logspace(
                0,
                2,
                d_model // 2
            )
        )

        self.fourier_scale = nn.Parameter(
            torch.ones(1)
        )

        # =================================================
        # input embedding
        # =================================================

        self.input_embed = nn.Sequential(

            nn.Linear(3, d_model),

            nn.ReLU(),

            nn.Linear(
                d_model,
                d_model
            )
        )

        # =================================================
        # FiLM
        # =================================================

        self.film = nn.Sequential(

            nn.Linear(4, d_model),

            nn.SiLU(),

            nn.Linear(d_model, d_model),

            nn.SiLU(),

            nn.Linear(
                d_model,
                d_model * 2
            )
        )

        # =================================================
        # fusion
        # =================================================

        self.fusion = nn.Linear(
            d_model * 2,
            d_model
        )

        # =================================================
        # transformer
        # =================================================

        encoder_layer = (
            nn.TransformerEncoderLayer(

                d_model=d_model,

                nhead=4,

                batch_first=True
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=2
            )
        )

        # =================================================
        # CLS token
        # =================================================

        self.cls_token = nn.Parameter(

            torch.randn(
                1,
                1,
                d_model
            )
        )

        # =================================================
        # Sentinel regression head
        # =================================================

        self.head = nn.Sequential(

            nn.Linear(
                d_model,
                d_model // 2
            ),

            nn.ReLU(),

            nn.Linear(
                d_model // 2,
                1
            )
        )

    # =====================================================
    # Fourier encoding
    # =====================================================

    def fourier_encoding(self, x):

        x = x.unsqueeze(-1)

        proj = (
            x
            * self.fourier_B
            * self.fourier_scale
        )

        return torch.cat([

            torch.sin(proj),

            torch.cos(proj)

        ], dim=-1)

    # =====================================================
    # spectral projection
    # =====================================================

    def spectral_project(
        self,
        z,
        wl,
        delta
    ):

        grid = (
            self.grid
            .to(z.device)
            .unsqueeze(0)
            .unsqueeze(0)
        )

        wl = wl.unsqueeze(-1)

        delta = delta.unsqueeze(-1)

        sigma = delta + 1e-6

        weight = torch.exp(

            -((wl - grid) ** 2)
            / (sigma ** 2)
        )

        weight = (
            weight
            / (
                weight.sum(
                    dim=1,
                    keepdim=True
                ) + 1e-8
            )
        )

        return (
            z.unsqueeze(2)
            * weight.unsqueeze(-1)
        ).sum(dim=1)

    # =====================================================
    # forward
    # =====================================================

    def forward(
        self,
        x,
        wl,
        delta,
        dtype,
        temp
    ):

        # -------------------------------------------------
        # input
        # -------------------------------------------------

        x = x.unsqueeze(-1)

        wl = wl.unsqueeze(-1)

        delta = delta.unsqueeze(-1)

        inp = torch.cat([
            x,
            wl,
            delta
        ], dim=-1)

        # -------------------------------------------------
        # input embed
        # -------------------------------------------------

        z = self.input_embed(inp)

        # -------------------------------------------------
        # FiLM
        # -------------------------------------------------

        cond = torch.stack([

            dtype.float(),

            temp,

            wl.mean(dim=1).squeeze(-1),

            delta.mean(dim=1).squeeze(-1)

        ], dim=1)

        gamma, beta = (
            self.film(cond)
            .chunk(2, dim=1)
        )

        z = (
            gamma.unsqueeze(1) * z
            + beta.unsqueeze(1)
        )

        # -------------------------------------------------
        # spectral projection
        # -------------------------------------------------

        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1)
        )

        # -------------------------------------------------
        # Fourier encoding
        # -------------------------------------------------

        grid = (
            self.grid
            .unsqueeze(0)
            .expand(
                z.size(0),
                -1
            )
        )

        pos = self.fourier_encoding(
            grid
        )

        z = torch.cat([
            z,
            pos
        ], dim=-1)

        # -------------------------------------------------
        # fusion
        # -------------------------------------------------

        z = self.fusion(z)

        # -------------------------------------------------
        # transformer
        # -------------------------------------------------

        B = z.size(0)

        cls = self.cls_token.expand(
            B,
            -1,
            -1
        )

        z = torch.cat([
            cls,
            z
        ], dim=1)

        z = self.encoder(z)

        z = z[:, 0]

        # -------------------------------------------------
        # head
        # -------------------------------------------------

        return self.head(z).squeeze()

# =========================================================
# LOAD PRETRAINED UAV->S2A MODEL
# =========================================================

def load_pretrained_weights(model):

    print("\nLoading UAV->S2A pretrained model...")

    pretrained = torch.load(
        PRETRAIN_PATH,
        map_location=DEVICE
    )

    model_dict = model.state_dict()

    # =====================================================
    # REMOVE HEAD
    # =====================================================

    pretrained_dict = {

        k:v for k,v in pretrained.items()

        if (
            k in model_dict
            and "head" not in k
        )
    }

    model_dict.update(
        pretrained_dict
    )

    model.load_state_dict(
        model_dict
    )

    print(
        f"Loaded "
        f"{len(pretrained_dict)} "
        f"parameters"
    )

# =========================================================
# EVALUATE
# =========================================================

def evaluate(
    model,
    loader
):

    model.eval()

    y_true = []

    y_pred = []

    with torch.no_grad():

        for batch in loader:

            pred = model(

                batch["x"].to(DEVICE),

                batch["lambda"].to(DEVICE),

                batch["delta"].to(DEVICE),

                batch["type"].to(DEVICE),

                batch["temp"].to(DEVICE)
            )

            y_true.extend(
                batch["y"].cpu().numpy()
            )

            y_pred.extend(
                pred.cpu().numpy()
            )

    y_true = np.array(y_true)

    y_pred = np.array(y_pred)

    mse = mean_squared_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return mse, rmse, mae, r2

def train():

    print("\n========================")
    print("Sentinel-2 Transfer")
    print("========================")

    # =====================================================
    # dataset
    # =====================================================

    dataset = SentinelDataset(S2_PATH)

    # =====================================================
    # folds
    # =====================================================

    import json

    with open(SPLIT_PATH, "r") as f:

        folds = json.load(f)

    results = []

    # =====================================================
    # 5-fold
    # =====================================================

    for fold_id in range(5):

        print(
            f"\n================ "
            f"FOLD {fold_id}"
            f" ================"
        )

        # -------------------------------------------------
        # split
        # -------------------------------------------------

        train_idx = np.array(
            folds[fold_id]["train_idx"]
        )

        test_idx = np.array(
            folds[fold_id]["test_idx"]
        )

        train_ds = torch.utils.data.Subset(
            dataset,
            train_idx
        )

        test_ds = torch.utils.data.Subset(
            dataset,
            test_idx
        )

        train_loader = DataLoader(

            train_ds,

            batch_size=32,

            shuffle=True
        )

        test_loader = DataLoader(

            test_ds,

            batch_size=32,

            shuffle=False
        )

        # =================================================
        # model
        # =================================================

        model = (
            SpectralFusionModel()
            .to(DEVICE)
        )

        # =================================================
        # load pretrained
        # =================================================

        load_pretrained_weights(model)

        criterion = nn.MSELoss()

        # =================================================
        # STAGE1
        # =================================================

        print("\nStage1: adapter training")

        # -------------------------------------------------
        # freeze all
        # -------------------------------------------------

        for p in model.parameters():

            p.requires_grad = False

        # -------------------------------------------------
        # unfreeze input_embed
        # -------------------------------------------------

        for p in model.input_embed.parameters():

            p.requires_grad = True

        # -------------------------------------------------
        # unfreeze FiLM
        # -------------------------------------------------

        for p in model.film.parameters():

            p.requires_grad = True

        # -------------------------------------------------
        # unfreeze head
        # -------------------------------------------------

        for p in model.head.parameters():

            p.requires_grad = True

        optimizer = torch.optim.Adam(

            filter(
                lambda p: p.requires_grad,
                model.parameters()
            ),

            lr=1e-3
        )

        best_r2 = -999

        best_stage1 = None

        wait = 0

        patience = 50

        # =================================================
        # stage1 training
        # =================================================

        for epoch in range(500):

            model.train()

            total_loss = 0

            for batch in train_loader:

                pred = model(

                    batch["x"].to(DEVICE),

                    batch["lambda"].to(DEVICE),

                    batch["delta"].to(DEVICE),

                    batch["type"].to(DEVICE),

                    batch["temp"].to(DEVICE)
                )

                loss = criterion(

                    pred,

                    batch["y"].to(DEVICE)
                )

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()

                total_loss += loss.item()

            mse, rmse, mae, r2 = evaluate(
                model,
                test_loader
            )

            print(

                f"[Stage1] "

                f"Epoch {epoch:03d} | "

                f"Loss {total_loss:.4f} | "

                f"RMSE {rmse:.4f} | "

                f"R2 {r2:.4f}"
            )

            # -------------------------------------------------
            # early stopping
            # -------------------------------------------------

            if r2 > best_r2:

                best_r2 = r2

                best_stage1 = copy.deepcopy(
                    model.state_dict()
                )

                wait = 0

            else:

                wait += 1

                if wait >= patience:

                    print(
                        "\nStage1 Early stopping"
                    )

                    break

        # =================================================
        # restore best stage1
        # =================================================

        model.load_state_dict(
            best_stage1
        )

        # =================================================
        # STAGE2
        # =================================================

        print("\nStage2: partial finetuning")

        # -------------------------------------------------
        # freeze all
        # -------------------------------------------------

        for p in model.parameters():

            p.requires_grad = False

        # -------------------------------------------------
        # train adapter
        # -------------------------------------------------

        for p in model.input_embed.parameters():

            p.requires_grad = True

        for p in model.film.parameters():

            p.requires_grad = True

        for p in model.head.parameters():

            p.requires_grad = True

        # -------------------------------------------------
        # unfreeze fusion
        # -------------------------------------------------

        for p in model.fusion.parameters():

            p.requires_grad = True

        # -------------------------------------------------
        # unfreeze encoder last layer
        # -------------------------------------------------

        for p in model.encoder.layers[1].parameters():

            p.requires_grad = True

        optimizer = torch.optim.Adam(

            filter(
                lambda p: p.requires_grad,
                model.parameters()
            ),

            lr=1e-4
        )

        best_r2 = -999

        best_model = None

        wait = 0

        patience = 50

        # =================================================
        # stage2 training
        # =================================================

        for epoch in range(500):

            model.train()

            total_loss = 0

            for batch in train_loader:

                pred = model(

                    batch["x"].to(DEVICE),

                    batch["lambda"].to(DEVICE),

                    batch["delta"].to(DEVICE),

                    batch["type"].to(DEVICE),

                    batch["temp"].to(DEVICE)
                )

                loss = criterion(

                    pred,

                    batch["y"].to(DEVICE)
                )

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()

                total_loss += loss.item()

            mse, rmse, mae, r2 = evaluate(
                model,
                test_loader
            )

            print(

                f"[Stage2] "

                f"Epoch {epoch:03d} | "

                f"Loss {total_loss:.4f} | "

                f"RMSE {rmse:.4f} | "

                f"R2 {r2:.4f}"
            )

            # -------------------------------------------------
            # early stopping
            # -------------------------------------------------

            if r2 > best_r2:

                best_r2 = r2

                best_model = copy.deepcopy(
                    model.state_dict()
                )

                wait = 0

                torch.save(

                    best_model,

                    os.path.join(

                        SAVE_DIR,

                        f"best_fold_{fold_id}.pth"
                    )
                )

            else:

                wait += 1

                if wait >= patience:

                    print(
                        "\nStage2 Early stopping"
                    )

                    break

        # =================================================
        # final evaluation
        # =================================================

        model.load_state_dict(
            best_model
        )

        mse, rmse, mae, r2 = evaluate(
            model,
            test_loader
        )

        print("\n========================")
        print(f"FOLD {fold_id} RESULT")
        print("========================")

        print(f"MSE  : {mse:.4f}")
        print(f"RMSE : {rmse:.4f}")
        print(f"MAE  : {mae:.4f}")
        print(f"R2   : {r2:.4f}")

        results.append([

            mse,

            rmse,

            mae,

            r2
        ])

    # =====================================================
    # summary
    # =====================================================

    results = np.array(results)

    print("\n========================")
    print("5-FOLD FINAL RESULT")
    print("========================")

    print(

        f"MSE  : "
        f"{results[:,0].mean():.4f} ± "
        f"{results[:,0].std():.4f}"
    )

    print(

        f"RMSE : "
        f"{results[:,1].mean():.4f} ± "
        f"{results[:,1].std():.4f}"
    )

    print(

        f"MAE  : "
        f"{results[:,2].mean():.4f} ± "
        f"{results[:,2].std():.4f}"
    )

    print(

        f"R2   : "
        f"{results[:,3].mean():.4f} ± "
        f"{results[:,3].std():.4f}"
    )

# =========================================================

if __name__ == "__main__":

    train()


Sentinel-2 Transfer

================ FOLD 0 ================

Loading UAV->S2A pretrained model...
Loaded 40 parameters

Stage1: adapter training


C:\Users\PC\AppData\Local\Temp\ipykernel_56944\1408777693.py:616: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load(


[Stage1] Epoch 000 | Loss 2242.5259 | RMSE 8.1791 | R2 -20.1914
[Stage1] Epoch 001 | Loss 2026.4222 | RMSE 7.7552 | R2 -18.0514
[Stage1] Epoch 002 | Loss 1784.1796 | RMSE 7.1282 | R2 -15.0953
[Stage1] Epoch 003 | Loss 1447.2761 | RMSE 6.0703 | R2 -10.6725
[Stage1] Epoch 004 | Loss 913.2511 | RMSE 4.3959 | R2 -5.1214
[Stage1] Epoch 005 | Loss 463.6634 | RMSE 2.8774 | R2 -1.6227
[Stage1] Epoch 006 | Loss 212.1765 | RMSE 2.0046 | R2 -0.2730
[Stage1] Epoch 007 | Loss 116.3353 | RMSE 1.8005 | R2 -0.0269
[Stage1] Epoch 008 | Loss 95.3525 | RMSE 1.6637 | R2 0.1232
[Stage1] Epoch 009 | Loss 85.6469 | RMSE 1.5729 | R2 0.2163
[Stage1] Epoch 010 | Loss 82.7436 | RMSE 1.5912 | R2 0.1979
[Stage1] Epoch 011 | Loss 75.9333 | RMSE 1.5838 | R2 0.2055
[Stage1] Epoch 012 | Loss 77.3268 | RMSE 1.5912 | R2 0.1979
[Stage1] Epoch 013 | Loss 79.6166 | RMSE 1.6211 | R2 0.1675
[Stage1] Epoch 014 | Loss 77.6496 | RMSE 1.6452 | R2 0.1426
[Stage1] Epoch 015 | Loss 76.6136 | RMSE 1.6020 | R2 0.1871
[Stage1] Epoch 0

C:\Users\PC\AppData\Local\Temp\ipykernel_56944\1408777693.py:616: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load(


[Stage1] Epoch 001 | Loss 1408.0870 | RMSE 6.2624 | R2 -11.9555
[Stage1] Epoch 002 | Loss 960.1785 | RMSE 4.9165 | R2 -6.9854
[Stage1] Epoch 003 | Loss 546.7668 | RMSE 3.4894 | R2 -3.0223
[Stage1] Epoch 004 | Loss 260.1353 | RMSE 2.3499 | R2 -0.8241
[Stage1] Epoch 005 | Loss 130.6969 | RMSE 1.8154 | R2 -0.0888
[Stage1] Epoch 006 | Loss 90.5683 | RMSE 1.7132 | R2 0.0304
[Stage1] Epoch 007 | Loss 89.6275 | RMSE 1.7079 | R2 0.0364
[Stage1] Epoch 008 | Loss 88.0310 | RMSE 1.7090 | R2 0.0351
[Stage1] Epoch 009 | Loss 88.9722 | RMSE 1.7030 | R2 0.0419
[Stage1] Epoch 010 | Loss 87.0938 | RMSE 1.7031 | R2 0.0418
[Stage1] Epoch 011 | Loss 88.2828 | RMSE 1.6991 | R2 0.0463
[Stage1] Epoch 012 | Loss 87.3400 | RMSE 1.6937 | R2 0.0523
[Stage1] Epoch 013 | Loss 87.9310 | RMSE 1.6910 | R2 0.0553
[Stage1] Epoch 014 | Loss 86.4494 | RMSE 1.6838 | R2 0.0634
[Stage1] Epoch 015 | Loss 86.4103 | RMSE 1.6736 | R2 0.0747
[Stage1] Epoch 016 | Loss 85.3469 | RMSE 1.6702 | R2 0.0785
[Stage1] Epoch 017 | Loss 85

...

[Stage1] Epoch 001 | Loss 1747.8682 | RMSE 7.1859 | R2 -17.9289
[Stage1] Epoch 002 | Loss 1342.7670 | RMSE 5.9931 | R2 -12.1665
[Stage1] Epoch 003 | Loss 855.2712 | RMSE 4.4677 | R2 -6.3169
[Stage1] Epoch 004 | Loss 431.0716 | RMSE 2.9415 | R2 -2.1718
[Stage1] Epoch 005 | Loss 189.2750 | RMSE 1.9406 | R2 -0.3805
[Stage1] Epoch 006 | Loss 105.8001 | RMSE 1.6438 | R2 0.0095
[Stage1] Epoch 007 | Loss 91.7496 | RMSE 1.6270 | R2 0.0296
[Stage1] Epoch 008 | Loss 90.8056 | RMSE 1.6224 | R2 0.0351
[Stage1] Epoch 009 | Loss 90.9486 | RMSE 1.6147 | R2 0.0443
[Stage1] Epoch 010 | Loss 88.7718 | RMSE 1.6139 | R2 0.0452
[Stage1] Epoch 011 | Loss 89.2735 | RMSE 1.6026 | R2 0.0585
[Stage1] Epoch 012 | Loss 88.7820 | RMSE 1.6056 | R2 0.0550
[Stage1] Epoch 013 | Loss 87.1525 | RMSE 1.5988 | R2 0.0629
[Stage1] Epoch 014 | Loss 86.8866 | RMSE 1.5961 | R2 0.0661
[Stage1] Epoch 015 | Loss 86.6432 | RMSE 1.5881 | R2 0.0755
[Stage1] Epoch 016 | Loss 86.4186 | RMSE 1.5823 | R2 0.0822
[Stage1] Epoch 017 | Loss

C:\Users\PC\AppData\Local\Temp\ipykernel_56944\1408777693.py:616: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load(


[Stage1] Epoch 001 | Loss 1954.4674 | RMSE 7.5215 | R2 -17.5454
[Stage1] Epoch 002 | Loss 1664.2048 | RMSE 6.8075 | R2 -14.1915
[Stage1] Epoch 003 | Loss 1309.9165 | RMSE 5.7480 | R2 -9.8306
[Stage1] Epoch 004 | Loss 873.5322 | RMSE 4.4578 | R2 -5.5143
[Stage1] Epoch 005 | Loss 506.9795 | RMSE 3.2186 | R2 -2.3959
[Stage1] Epoch 006 | Loss 261.3642 | RMSE 2.2693 | R2 -0.6881
[Stage1] Epoch 007 | Loss 142.2930 | RMSE 1.8197 | R2 -0.0855
[Stage1] Epoch 008 | Loss 99.1922 | RMSE 1.7139 | R2 0.0371
[Stage1] Epoch 009 | Loss 90.3645 | RMSE 1.7098 | R2 0.0416
[Stage1] Epoch 010 | Loss 87.5827 | RMSE 1.7090 | R2 0.0425
[Stage1] Epoch 011 | Loss 87.2210 | RMSE 1.7170 | R2 0.0336
[Stage1] Epoch 012 | Loss 88.4737 | RMSE 1.7096 | R2 0.0419
[Stage1] Epoch 013 | Loss 87.6544 | RMSE 1.7109 | R2 0.0404
[Stage1] Epoch 014 | Loss 87.6804 | RMSE 1.6988 | R2 0.0540
[Stage1] Epoch 015 | Loss 86.7066 | RMSE 1.6897 | R2 0.0641
[Stage1] Epoch 016 | Loss 85.3955 | RMSE 1.6849 | R2 0.0694
[Stage1] Epoch 017 | 

C:\Users\PC\AppData\Local\Temp\ipykernel_56944\1408777693.py:616: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load(


[Stage1] Epoch 001 | Loss 2023.8341 | RMSE 8.0205 | R2 -22.4598
[Stage1] Epoch 002 | Loss 1956.1446 | RMSE 7.6578 | R2 -20.3863
[Stage1] Epoch 003 | Loss 1663.5380 | RMSE 6.8486 | R2 -16.1051
[Stage1] Epoch 004 | Loss 1262.8284 | RMSE 5.6449 | R2 -10.6207
[Stage1] Epoch 005 | Loss 812.9922 | RMSE 4.3805 | R2 -5.9981
[Stage1] Epoch 006 | Loss 471.0784 | RMSE 3.1861 | R2 -2.7019
[Stage1] Epoch 007 | Loss 254.6012 | RMSE 2.3187 | R2 -0.9606
[Stage1] Epoch 008 | Loss 147.0524 | RMSE 1.8426 | R2 -0.2381
[Stage1] Epoch 009 | Loss 107.7428 | RMSE 1.6729 | R2 -0.0206
[Stage1] Epoch 010 | Loss 90.3828 | RMSE 1.5955 | R2 0.0717
[Stage1] Epoch 011 | Loss 84.9688 | RMSE 1.5720 | R2 0.0988
[Stage1] Epoch 012 | Loss 82.7089 | RMSE 1.5539 | R2 0.1194
[Stage1] Epoch 013 | Loss 81.2571 | RMSE 1.5281 | R2 0.1484
[Stage1] Epoch 014 | Loss 81.5955 | RMSE 1.5152 | R2 0.1628
[Stage1] Epoch 015 | Loss 79.9040 | RMSE 1.5033 | R2 0.1758
[Stage1] Epoch 016 | Loss 78.8122 | RMSE 1.5042 | R2 0.1749
[Stage1] Epoch

### UAV_transfer

#### UAV_model_acc_test

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import copy
import torch.nn.functional as F

DATA_DIR = r"\data\input"
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")
model_save_dir = os.path.join(DATA_DIR, "Sentinel2_transfer/UAV/PI-Transformer")
os.makedirs(model_save_dir, exist_ok=True)

# =========================
# ⭐ dataset config
# =========================
selected_datasets = ["uav"]

DATASET_CONFIG = {
    "uav": (0, "UAV.csv"),
    "sentinel": (1, "Sentinel-2.csv"),
    "landsat": (2, "Landsat-8.csv"),
}

# =========================
# 0. wavelength normalization
# =========================
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002

def normalize_wavelength(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

def normalize_delta(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

# =========================
# ⭐ split loader
# =========================
def load_split(path, fold_id=0):
    with open(path, "r") as f:
        folds = json.load(f)

    split = folds[fold_id]
    return (
        np.array(split["train_idx"]),
        np.array(split["test_idx"])
    )

# =========================
# Dataset
# =========================
class SpectralDataset(Dataset):
    def __init__(self, file_path, dtype, x_mean, x_std):
        df = pd.read_csv(file_path, encoding='gbk')
        self.dtype = dtype

        temp = df.iloc[:, -2].values.astype(np.float32)
        self.x = df.iloc[:, :-2].values.astype(np.float32)
        self.y = df.iloc[:, -1].values.astype(np.float32)

        # NDWI filter
        if dtype == 1:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 2:
            ndwi = (df['SR_B3'] - df['SR_B5']) / (df['SR_B3'] + df['SR_B5'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # clean
        mask = np.all(self.x >= 0, axis=1)
        self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # global norm
        self.x = (self.x - x_mean) / (x_std + 1e-6)

        # wavelength
        if dtype == 0:
            self.lambda_ = np.arange(350, 1003, 4).astype(np.float32)
            self.delta = np.full_like(self.lambda_, 4.0)
            self.group_id = np.arange(len(self.x)) // 10

        elif dtype == 1:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),(740,15),
                     (783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        else:
            bands = [(443,20),(482,65),(561,75),(655,50),(865,40)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        self.lambda_ = normalize_wavelength(self.lambda_)
        self.delta = normalize_delta(self.delta)

        self.temp = (temp - temp.mean()) / (temp.std() + 1e-6)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(self.x[idx]),
            "lambda": torch.tensor(self.lambda_),
            "delta": torch.tensor(self.delta),
            "y": torch.tensor(self.y[idx]),
            "type": torch.tensor(self.dtype),
            "temp": torch.tensor(self.temp[idx])
        }

# =========================
# Model
# =========================
class SpectralFusionModel(nn.Module):
    def __init__(self, d_model=64, grid_size=64):
        super().__init__()

        self.d_model = d_model

        # =========================
        # 1. fixed spectral grid
        # =========================
        self.register_buffer(
            "grid",
            torch.linspace(0, 1, grid_size)
        )

        # =========================
        # 2. Fourier Encoding
        # =========================
        self.fourier_B = nn.Parameter(
            torch.logspace(0, 2, d_model // 2)
        )
        self.fourier_scale = nn.Parameter(torch.ones(1))

        # fusion layer to combine spectral and positional embeddings
        self.fusion = nn.Linear(d_model * 2, d_model)

        # =========================
        # 3. Input embedding
        # =========================
        self.input_embed = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.temp_embed = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # =========================
        # 3.5 learning spectral weight network
        # =========================
        self.spectral_weight_net = nn.Sequential(
            nn.Linear(4, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
            nn.SiLU(),
            nn.Linear(d_model, 1)
        )

        # =========================
        # 4. FiLM conditioning
        # =========================
        self.film = nn.Sequential(
            nn.Linear(4, d_model // 2),
            nn.SiLU(),
            nn.Linear(d_model // 2, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model * 2)
        )

        # =========================
        # 5. Transformer Encoder
        # =========================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # =========================
        # 7. Regression head
        # =========================
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    # =========================
    # Fourier Encoding
    # =========================
    def fourier_encoding(self, x):
        x = x.unsqueeze(-1)  # (B, N, 1)

        proj = x * self.fourier_B * self.fourier_scale

        return torch.cat([
            torch.sin(proj),
            torch.cos(proj)
        ], dim=-1)

    # =========================
    # spectral projection
    # =========================
    def spectral_project(self, z, wl, delta, dtype):

        grid = self.grid.to(z.device).unsqueeze(0).unsqueeze(0)

        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)
        dtype = dtype.unsqueeze(-1).unsqueeze(-1).float()

        sigma = delta + 1e-6

        weight = torch.exp(-((wl - grid) ** 2) / (sigma ** 2))

        B, N, _ = wl.shape
        G = grid.size(-1)

        grid_exp = grid.expand(B, N, G)
        wl_exp = wl.expand(B, N, G)
        delta_exp = delta.expand(B, N, G)
        dtype_exp = dtype.expand(B, N, G)

        weight2_input = torch.stack(
            [dtype_exp, wl_exp, delta_exp, grid_exp],
            dim=-1
        )
        weight2 = torch.sigmoid(self.spectral_weight_net(weight2_input)).squeeze(-1)

        weight = weight * weight2
        weight = weight / (weight.sum(dim=1, keepdim=True) + 1e-8)

        return (z.unsqueeze(2) * weight.unsqueeze(-1)).sum(dim=1)

    # =========================
    # forward
    # =========================
    def forward(self, x, wl, delta, dtype, temp):

        # -------------------------
        # 1. input embedding
        # -------------------------
        x = x.unsqueeze(-1)
        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        inp = torch.cat([x, wl, delta], dim=-1)
        z = self.input_embed(inp)

        # -------------------------
        # 2. FiLM conditioning
        # -------------------------
        cond = torch.stack([
            dtype.float(),
            temp,
            wl.mean(dim=1).squeeze(-1),
            delta.mean(dim=1).squeeze(-1)
        ], dim=1)

        gamma, beta = self.film(cond).chunk(2, dim=1)
        z = gamma.unsqueeze(1) * z + beta.unsqueeze(1)

        # -------------------------
        # 3. spectral projection
        # -------------------------
        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1),
            dtype
        )

        # -------------------------
        # 4. Fourier positional encoding
        # -------------------------
        grid = self.grid.unsqueeze(0).expand(z.size(0), -1)

        pos = self.fourier_encoding(grid)

        z = torch.cat([z, pos], dim=-1)
        z = self.fusion(z)

        # -------------------------
        # 5. Transformer + CLS token pooling
        # -------------------------
        B = z.size(0)
        t = self.temp_embed(temp.unsqueeze(-1)).unsqueeze(1)
        z = torch.cat([t,  z], dim=1)          # (B,N+1,d)

        z = self.encoder(z)

        z = z[:, 0]  # ⭐ CLS token output

        # -------------------------
        # 6. regression head
        # -------------------------
        return self.head(z).squeeze()

# =========================
# Loss
# =========================
def r2_aware_loss(pred, target):
    mse = torch.mean((pred - target) ** 2)

    pred_c = pred - pred.mean()
    tgt_c = target - target.mean()

    corr = torch.sum(pred_c * tgt_c) / (
        torch.sqrt(torch.sum(pred_c ** 2)) *
        torch.sqrt(torch.sum(tgt_c ** 2)) + 1e-8
    )

    return mse + 0.5 * (1 - corr)

# =========================
# ⭐ evaluation
# =========================
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in loader:
            pred = model(
                batch["x"].to(device),
                batch["lambda"].to(device),
                batch["delta"].to(device),
                batch["type"].to(device),
                batch["temp"].to(device)
            )

            y_true.extend(batch["y"].cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mse, rmse, mae, r2


# =========================
# ⭐ train with early stopping
# =========================
def train():

    device = "cuda" if torch.cuda.is_available() else "cpu"

    paths = [
        (DATASET_CONFIG[name][0],
         os.path.join(DATA_DIR, DATASET_CONFIG[name][1]))
        for name in selected_datasets
    ]

    # =========================
    # global normalization
    # =========================
    all_x = []
    for _, path in paths:
        df = pd.read_csv(path, encoding='gbk')
        all_x.append(df.iloc[:, :-2].values.astype(np.float32))

    all_x = np.vstack(all_x)
    x_mean = all_x.mean(axis=0)
    x_std = all_x.std(axis=0) + 1e-6

    dataset = SpectralDataset(paths[0][1], paths[0][0], x_mean, x_std)

    # =========================
    # ⭐ 5-fold split
    # =========================
    folds = json.load(open(SPLIT_PATH))

    results = []

    for fold_id in range(5):

        print(f"\n================ FOLD {fold_id} ================")

        train_idx = np.array(folds[fold_id]["train_idx"])
        test_idx  = np.array(folds[fold_id]["test_idx"])

        train_ds = Subset(dataset, train_idx)
        test_ds  = Subset(dataset, test_idx)

        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=16)

        model = SpectralFusionModel().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=3e-4)

        # =========================
        # ⭐ early stopping
        # =========================
        best_loss = float("inf")
        patience = 50
        wait = 0
        best_model = None

        for epoch in range(500):

            model.train()
            total_loss = 0

            for batch in train_loader:
                pred = model(
                    batch["x"].to(device),
                    batch["lambda"].to(device),
                    batch["delta"].to(device),
                    batch["type"].to(device),
                    batch["temp"].to(device)
                )

                loss = r2_aware_loss(pred, batch["y"].to(device))

                opt.zero_grad()
                loss.backward()
                opt.step()

                total_loss += loss.item()

            # =========================
            # ⭐ validation = test here
            # =========================
            mse, rmse, mae, r2 = evaluate(model, test_loader, device)

            print(
                f"Epoch {epoch:03d} | "
                f"TrainLoss {total_loss:.4f} | "
                f"Test RMSE {rmse:.4f} R2 {r2:.4f}"
            )

            # =========================
            # ⭐ save best model
            # =========================
            if rmse < best_loss:
                best_loss = rmse
                best_model = copy.deepcopy(model.state_dict())
                wait = 0

                torch.save(
                    best_model,
                    os.path.join(model_save_dir, f"best_model_fold{fold_id}.pth")
                )
            else:
                wait += 1
                if wait >= patience:
                    print("Early stopping triggered")
                    break

        # =========================
        # ⭐ final evaluation
        # =========================
        model.load_state_dict(best_model)

        mse, rmse, mae, r2 = evaluate(model, test_loader, device)

        print(f"\n[FOLD {fold_id} FINAL]")
        print(f"MSE: {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE: {mae:.4f}")
        print(f"R2: {r2:.4f}")

        results.append([mse, rmse, mae, r2])

    # =========================
    # ⭐ overall result
    # =========================
    results = np.array(results)

    print("\n================ FINAL 5-FOLD RESULT ================")
    print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
    print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
    print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
    print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

if __name__ == "__main__":
    train()


================ FOLD 0 ================
Epoch 000 | TrainLoss 1306.4492 | Test RMSE 5.8593 R2 -12.1819
Epoch 001 | TrainLoss 919.4186 | Test RMSE 4.6947 R2 -7.4625
Epoch 002 | TrainLoss 571.0932 | Test RMSE 3.3671 R2 -3.3532
Epoch 003 | TrainLoss 295.1157 | Test RMSE 2.1774 R2 -0.8204
Epoch 004 | TrainLoss 147.1730 | Test RMSE 1.5106 R2 0.1238
Epoch 005 | TrainLoss 76.4818 | Test RMSE 1.4259 R2 0.2194
Epoch 006 | TrainLoss 59.9915 | Test RMSE 1.4625 R2 0.1788
Epoch 007 | TrainLoss 60.1597 | Test RMSE 1.5219 R2 0.1107
Epoch 008 | TrainLoss 58.1692 | Test RMSE 1.4977 R2 0.1388
Epoch 009 | TrainLoss 57.1203 | Test RMSE 1.4427 R2 0.2009
Epoch 010 | TrainLoss 55.5106 | Test RMSE 1.4076 R2 0.2393
Epoch 011 | TrainLoss 56.1273 | Test RMSE 1.6115 R2 0.0029
Epoch 012 | TrainLoss 57.5488 | Test RMSE 1.5055 R2 0.1297
Epoch 013 | TrainLoss 56.1168 | Test RMSE 1.5074 R2 0.1276
Epoch 014 | TrainLoss 51.6464 | Test RMSE 1.5198 R2 0.1131
Epoch 015 | TrainLoss 50.8966 | Test RMSE 1.3838 R2 0.2647
Epo

#### UAV_model_train

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import copy
import torch.nn.functional as F

DATA_DIR = r"E:\LYQ\work\250916remote_model\data\input"
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")
model_save_dir = os.path.join(DATA_DIR, "Sentinel2_transfer/UAV/train_full/PI-Transformer")
os.makedirs(model_save_dir, exist_ok=True)

# =========================
# ⭐ dataset config
# =========================
selected_datasets = ["uav"]

DATASET_CONFIG = {
    "uav": (0, "UAV.csv"),
    "sentinel": (1, "Sentinel-2.csv"),
    "landsat": (2, "Landsat-8.csv"),
    "uav2s2a": (3, "UAV2S2A.csv")

}

# =========================
# 0. wavelength normalization
# =========================
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002

def normalize_wavelength(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

def normalize_delta(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

# =========================
# Dataset
# =========================
class SpectralDataset(Dataset):
    def __init__(self, file_path, dtype, x_mean, x_std):
        df = pd.read_csv(file_path, encoding='gbk')
        self.dtype = dtype

        temp = df.iloc[:, -2].values.astype(np.float32)
        self.x = df.iloc[:, :-2].values.astype(np.float32)
        self.y = df.iloc[:, -1].values.astype(np.float32)

        # NDWI filter
        if dtype == 1:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 2:
            ndwi = (df['SR_B3'] - df['SR_B5']) / (df['SR_B3'] + df['SR_B5'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 3:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # clean
        mask = np.all(self.x >= 0, axis=1)
        self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # global norm
        self.x = (self.x - x_mean) / (x_std + 1e-6)

        # wavelength
        if dtype == 0:
            self.lambda_ = np.arange(350, 1003, 4).astype(np.float32)
            self.delta = np.full_like(self.lambda_, 4.0)
            self.group_id = np.arange(len(self.x)) // 10

        elif dtype == 1:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),
                     (740,15),(783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        elif dtype == 2:
            bands = [(443,20),(482,65),(561,75),(655,50),(865,40)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        elif dtype == 3:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),
                     (740,15),(783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        self.lambda_ = normalize_wavelength(self.lambda_)
        self.delta = normalize_delta(self.delta)

        self.temp = (temp - temp.mean()) / (temp.std() + 1e-6)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(self.x[idx]),
            "lambda": torch.tensor(self.lambda_),
            "delta": torch.tensor(self.delta),
            "y": torch.tensor(self.y[idx]),
            "type": torch.tensor(self.dtype),
            "temp": torch.tensor(self.temp[idx])
        }

# =========================
# Model
# =========================
class SpectralFusionModel(nn.Module):
    def __init__(self, d_model=64, grid_size=64):
        super().__init__()

        self.d_model = d_model

        # =========================
        # 1. fixed spectral grid
        # =========================
        self.register_buffer(
            "grid",
            torch.linspace(0, 1, grid_size)
        )

        # =========================
        # 2. Fourier Encoding
        # =========================
        self.fourier_B = nn.Parameter(
            torch.logspace(0, 2, d_model // 2)
        )
        self.fourier_scale = nn.Parameter(torch.ones(1))

        # fusion layer to combine spectral and positional embeddings
        self.fusion = nn.Linear(d_model * 2, d_model)

        # =========================
        # 3. Input embedding
        # =========================
        self.input_embed = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.temp_embed = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # =========================
        # 3.5 learning spectral weight network
        # =========================
        self.spectral_weight_net = nn.Sequential(
            nn.Linear(4, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
            nn.SiLU(),
            nn.Linear(d_model, 1)
        )

        # =========================
        # 4. FiLM conditioning
        # =========================
        self.film = nn.Sequential(
            nn.Linear(4, d_model // 2),
            nn.SiLU(),
            nn.Linear(d_model // 2, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model * 2)
        )

        # =========================
        # 5. Transformer Encoder
        # =========================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # =========================
        # 7. Regression head
        # =========================
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    # =========================
    # Fourier Encoding
    # =========================
    def fourier_encoding(self, x):
        x = x.unsqueeze(-1)  # (B, N, 1)

        proj = x * self.fourier_B * self.fourier_scale

        return torch.cat([
            torch.sin(proj),
            torch.cos(proj)
        ], dim=-1)

    # =========================
    # spectral projection
    # =========================
    def spectral_project(self, z, wl, delta, dtype):

        grid = self.grid.to(z.device).unsqueeze(0).unsqueeze(0)

        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)
        dtype = dtype.unsqueeze(-1).unsqueeze(-1).float()

        sigma = delta + 1e-6

        weight = torch.exp(-((wl - grid) ** 2) / (sigma ** 2))

        B, N, _ = wl.shape
        G = grid.size(-1)

        grid_exp = grid.expand(B, N, G)
        wl_exp = wl.expand(B, N, G)
        delta_exp = delta.expand(B, N, G)
        dtype_exp = dtype.expand(B, N, G)

        weight2_input = torch.stack(
            [dtype_exp, wl_exp, delta_exp, grid_exp],
            dim=-1
        )
        weight2 = torch.sigmoid(self.spectral_weight_net(weight2_input)).squeeze(-1)

        weight = weight * weight2
        weight = weight / (weight.sum(dim=1, keepdim=True) + 1e-8)

        return (z.unsqueeze(2) * weight.unsqueeze(-1)).sum(dim=1)

    # =========================
    # forward
    # =========================
    def forward(self, x, wl, delta, dtype, temp):

        # -------------------------
        # 1. input embedding
        # -------------------------
        x = x.unsqueeze(-1)
        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        inp = torch.cat([x, wl, delta], dim=-1)
        z = self.input_embed(inp)

        # -------------------------
        # 2. FiLM conditioning
        # -------------------------
        cond = torch.stack([
            dtype.float(),
            temp,
            wl.mean(dim=1).squeeze(-1),
            delta.mean(dim=1).squeeze(-1)
        ], dim=1)

        gamma, beta = self.film(cond).chunk(2, dim=1)
        z = gamma.unsqueeze(1) * z + beta.unsqueeze(1)

        # -------------------------
        # 3. spectral projection
        # -------------------------
        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1),
            dtype
        )

        # -------------------------
        # 4. Fourier positional encoding
        # -------------------------
        grid = self.grid.unsqueeze(0).expand(z.size(0), -1)

        pos = self.fourier_encoding(grid)

        z = torch.cat([z, pos], dim=-1)
        z = self.fusion(z)

        # -------------------------
        # 5. Transformer + CLS token pooling
        # -------------------------
        B = z.size(0)
        t = self.temp_embed(temp.unsqueeze(-1)).unsqueeze(1)
        z = torch.cat([t,  z], dim=1)          # (B,N+1,d)

        z = self.encoder(z)

        z = z[:, 0]  # ⭐ CLS token output

        # -------------------------
        # 6. regression head
        # -------------------------
        return self.head(z).squeeze()

# =========================
# Loss
# =========================
def r2_aware_loss(pred, target):
    mse = torch.mean((pred - target) ** 2)

    pred_c = pred - pred.mean()
    tgt_c = target - target.mean()

    corr = torch.sum(pred_c * tgt_c) / (
        torch.sqrt(torch.sum(pred_c ** 2)) *
        torch.sqrt(torch.sum(tgt_c ** 2)) + 1e-8
    )

    return mse + 0.5 * (1 - corr)

# =========================
# ⭐ evaluation
# =========================
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in loader:
            pred = model(
                batch["x"].to(device),
                batch["lambda"].to(device),
                batch["delta"].to(device),
                batch["type"].to(device),
                batch["temp"].to(device)
            )

            y_true.extend(batch["y"].cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mse, rmse, mae, r2
# =========================
# ⭐ train on full dataset
# =========================
def train():

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # =========================
    # save dir
    # =========================
    save_dir = os.path.join(
        DATA_DIR,
        "UAV2S2A/train_full/PI-Transformer"
    )
    os.makedirs(save_dir, exist_ok=True)

    paths = [
        (DATASET_CONFIG[name][0],
         os.path.join(DATA_DIR, DATASET_CONFIG[name][1]))
        for name in selected_datasets
    ]

    # =========================
    # global normalization
    # =========================
    all_x = []

    for _, path in paths:
        df = pd.read_csv(path, encoding='gbk')
        all_x.append(df.iloc[:, :-2].values.astype(np.float32))

    all_x = np.vstack(all_x)

    x_mean = all_x.mean(axis=0)
    x_std = all_x.std(axis=0) + 1e-6

    # =========================
    # full dataset
    # =========================
    dataset = SpectralDataset(
        paths[0][1],
        paths[0][0],
        x_mean,
        x_std
    )

    train_loader = DataLoader(
        dataset,
        batch_size=16,
        shuffle=True
    )

    # =========================
    # model
    # =========================
    model = SpectralFusionModel().to(device)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=3e-4
    )

    # =========================
    # train
    # =========================
    epochs = 200

    for epoch in range(epochs):

        model.train()

        total_loss = 0

        for batch in train_loader:

            pred = model(
                batch["x"].to(device),
                batch["lambda"].to(device),
                batch["delta"].to(device),
                batch["type"].to(device),
                batch["temp"].to(device)
            )

            loss = r2_aware_loss(
                pred,
                batch["y"].to(device)
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()

        # =========================
        # evaluate on training set
        # =========================
        mse, rmse, mae, r2 = evaluate(
            model,
            train_loader,
            device
        )

        print(
            f"Epoch {epoch:03d} | "
            f"TrainLoss {total_loss:.4f} | "
            f"Train RMSE {rmse:.4f} | "
            f"Train MAE {mae:.4f} | "
            f"Train R2 {r2:.4f}"
        )

    # =========================
    # save final model
    # =========================
    model_path = os.path.join(
        model_save_dir,
        "PI_Transformer_full.pth"
    )

    torch.save(model.state_dict(), model_path)

    print("\nTraining Finished")
    print(f"Model saved to:\n{model_path}")

if __name__ == "__main__":
    train()

Epoch 000 | TrainLoss 1313.2269 | Train RMSE 4.9013 | Train MAE 4.5439 | Train R2 -6.1176
Epoch 001 | TrainLoss 648.5237 | Train RMSE 3.1724 | Train MAE 2.7625 | Train R2 -1.9819
Epoch 002 | TrainLoss 267.6470 | Train RMSE 1.9950 | Train MAE 1.7085 | Train R2 -0.1792
Epoch 003 | TrainLoss 148.6555 | Train RMSE 1.8369 | Train MAE 1.4989 | Train R2 0.0002
Epoch 004 | TrainLoss 134.7960 | Train RMSE 1.7728 | Train MAE 1.4522 | Train R2 0.0688
Epoch 005 | TrainLoss 96.6636 | Train RMSE 1.4232 | Train MAE 1.1732 | Train R2 0.3999
Epoch 006 | TrainLoss 74.9659 | Train RMSE 1.3668 | Train MAE 1.1116 | Train R2 0.4465
Epoch 007 | TrainLoss 72.7994 | Train RMSE 1.3420 | Train MAE 1.1023 | Train R2 0.4664
Epoch 008 | TrainLoss 71.1670 | Train RMSE 1.3222 | Train MAE 1.0891 | Train R2 0.4820
Epoch 009 | TrainLoss 73.8584 | Train RMSE 1.3232 | Train MAE 1.0858 | Train R2 0.4812
Epoch 010 | TrainLoss 68.7997 | Train RMSE 1.2709 | Train MAE 1.0351 | Train R2 0.5215
Epoch 011 | TrainLoss 63.9654 | Tr

#### transfer

In [ ]:
import os
import copy
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from torch.utils.data import (
    Dataset,
    DataLoader,
    random_split
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# =========================================================
# PATH
# =========================================================

DATA_DIR = r"\data\input"
SPLIT_PATH = os.path.join(DATA_DIR, "Sentinel-2/Sentinel-2_splits.json")

S2_PATH = os.path.join(
    DATA_DIR,
    "Sentinel-2.csv"
)

# =========================================================
# UAV -> S2A pretrained model
# =========================================================

PRETRAIN_PATH = (
    r"\data\input"
    r"\Sentinel2_transfer/UAV/train_full/PI-Transformer"
    r"\PI_Transformer_full.pth"
)

SAVE_DIR = os.path.join(
    DATA_DIR,
    "Sentinel2_transfer"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# =========================================================
# GLOBAL NORMALIZATION
# =========================================================

GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002.0

def normalize_wavelength(x):

    return (
        (x - GLOBAL_LAMBDA_MIN)
        / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)
    )

# =========================================================
# DATASET
# =========================================================

class SentinelDataset(Dataset):

    def __init__(self, csv_path):

        df = pd.read_csv(
            csv_path,
            encoding="gbk"
        )

        # -------------------------------------------------
        # Sentinel-2 bands
        # -------------------------------------------------

        self.band_names = [
            "B1",
            "B2",
            "B3",
            "B4",
            "B5",
            "B6",
            "B7",
            "B8",
            "B8A",
            "B9"
        ]

        # -------------------------------------------------
        # center wavelength
        # -------------------------------------------------

        self.lambda_ = np.array([
            443,
            490,
            560,
            665,
            705,
            740,
            783,
            842,
            865,
            945

        ], dtype=np.float32)

        # -------------------------------------------------
        # bandwidth
        # -------------------------------------------------

        self.delta = np.array([
            20,
            65,
            35,
            30,
            15,
            15,
            20,
            105,
            20,
            20

        ], dtype=np.float32)

        # -------------------------------------------------
        # spectral input
        # -------------------------------------------------

        self.x = df[
            self.band_names
        ].values.astype(np.float32)

        # -------------------------------------------------
        # target
        # -------------------------------------------------

        self.y = (
            df.iloc[:, -1]
            .values
            .astype(np.float32)
        )

        # -------------------------------------------------
        # water temperature
        # -------------------------------------------------

        temp = (
            df.iloc[:, -2]
            .values
            .astype(np.float32)
        )

        # -------------------------------------------------
        # NDWI filter
        # -------------------------------------------------

        ndwi = (
            df["B3"] - df["B8"]
        ) / (
            df["B3"] + df["B8"] + 1e-8
        )

        mask = ndwi > 0.1

        self.x = self.x[mask]

        self.y = self.y[mask]

        temp = temp[mask]

        # -------------------------------------------------
        # remove invalid
        # -------------------------------------------------

        mask = np.all(
            self.x >= 0,
            axis=1
        )

        self.x = self.x[mask]

        self.y = self.y[mask]

        temp = temp[mask]

        # -------------------------------------------------
        # normalize spectral
        # -------------------------------------------------

        self.x_mean = self.x.mean(axis=0)

        self.x_std = (
            self.x.std(axis=0)
            + 1e-6
        )

        self.x = (
            self.x - self.x_mean
        ) / self.x_std

        # -------------------------------------------------
        # normalize temperature
        # -------------------------------------------------

        self.temp = (
            temp - temp.mean()
        ) / (
            temp.std() + 1e-6
        )

        # -------------------------------------------------
        # normalize wavelength
        # -------------------------------------------------

        self.lambda_ = normalize_wavelength(
            self.lambda_
        )

        self.delta = normalize_wavelength(
            self.delta
        )

    def __len__(self):

        return len(self.x)

    def __getitem__(self, idx):

        return {

            "x": torch.tensor(
                self.x[idx]
            ),

            "lambda": torch.tensor(
                self.lambda_
            ),

            "delta": torch.tensor(
                self.delta
            ),

            "temp": torch.tensor(
                self.temp[idx]
            ),

            # Sentinel type = 1
            "type": torch.tensor(1),

            "y": torch.tensor(
                self.y[idx]
            )
        }

# =========================
# Model
# =========================
class SpectralFusionModel(nn.Module):
    def __init__(self, d_model=64, grid_size=64):
        super().__init__()

        self.d_model = d_model

        # =========================
        # 1. fixed spectral grid
        # =========================
        self.register_buffer(
            "grid",
            torch.linspace(0, 1, grid_size)
        )

        # =========================
        # 2. Fourier Encoding
        # =========================
        self.fourier_B = nn.Parameter(
            torch.logspace(0, 2, d_model // 2)
        )
        self.fourier_scale = nn.Parameter(torch.ones(1))

        # fusion layer to combine spectral and positional embeddings
        self.fusion = nn.Linear(d_model * 2, d_model)

        # =========================
        # 3. Input embedding
        # =========================
        self.input_embed = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.temp_embed = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # =========================
        # 3.5 learning spectral weight network
        # =========================
        self.spectral_weight_net = nn.Sequential(
            nn.Linear(4, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
            nn.SiLU(),
            nn.Linear(d_model, 1)
        )

        # =========================
        # 4. FiLM conditioning
        # =========================
        self.film = nn.Sequential(
            nn.Linear(4, d_model // 2),
            nn.SiLU(),
            nn.Linear(d_model // 2, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model * 2)
        )

        # =========================
        # 5. Transformer Encoder
        # =========================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # =========================
        # 7. Regression head
        # =========================
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    # =========================
    # Fourier Encoding
    # =========================
    def fourier_encoding(self, x):
        x = x.unsqueeze(-1)  # (B, N, 1)

        proj = x * self.fourier_B * self.fourier_scale

        return torch.cat([
            torch.sin(proj),
            torch.cos(proj)
        ], dim=-1)

    # =========================
    # spectral projection
    # =========================
    def spectral_project(self, z, wl, delta, dtype):

        grid = self.grid.to(z.device).unsqueeze(0).unsqueeze(0)

        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)
        dtype = dtype.unsqueeze(-1).unsqueeze(-1).float()

        sigma = delta + 1e-6

        weight = torch.exp(-((wl - grid) ** 2) / (sigma ** 2))

        B, N, _ = wl.shape
        G = grid.size(-1)

        grid_exp = grid.expand(B, N, G)
        wl_exp = wl.expand(B, N, G)
        delta_exp = delta.expand(B, N, G)
        dtype_exp = dtype.expand(B, N, G)

        weight2_input = torch.stack(
            [dtype_exp, wl_exp, delta_exp, grid_exp],
            dim=-1
        )
        weight2 = torch.sigmoid(self.spectral_weight_net(weight2_input)).squeeze(-1)

        weight = weight * weight2
        weight = weight / (weight.sum(dim=1, keepdim=True) + 1e-8)

        return (z.unsqueeze(2) * weight.unsqueeze(-1)).sum(dim=1)

    # =========================
    # forward
    # =========================
    def forward(self, x, wl, delta, dtype, temp):

        # -------------------------
        # 1. input embedding
        # -------------------------
        x = x.unsqueeze(-1)
        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        inp = torch.cat([x, wl, delta], dim=-1)
        z = self.input_embed(inp)

        # -------------------------
        # 2. FiLM conditioning
        # -------------------------
        cond = torch.stack([
            dtype.float(),
            temp,
            wl.mean(dim=1).squeeze(-1),
            delta.mean(dim=1).squeeze(-1)
        ], dim=1)

        gamma, beta = self.film(cond).chunk(2, dim=1)
        z = gamma.unsqueeze(1) * z + beta.unsqueeze(1)

        # -------------------------
        # 3. spectral projection
        # -------------------------
        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1),
            dtype
        )

        # -------------------------
        # 4. Fourier positional encoding
        # -------------------------
        grid = self.grid.unsqueeze(0).expand(z.size(0), -1)

        pos = self.fourier_encoding(grid)

        z = torch.cat([z, pos], dim=-1)
        z = self.fusion(z)

        # -------------------------
        # 5. Transformer + CLS token pooling
        # -------------------------
        B = z.size(0)
        t = self.temp_embed(temp.unsqueeze(-1)).unsqueeze(1)
        z = torch.cat([t,  z], dim=1)          # (B,N+1,d)

        z = self.encoder(z)

        z = z[:, 0]  # ⭐ CLS token output

        # -------------------------
        # 6. regression head
        # -------------------------
        return self.head(z).squeeze()

# =========================================================
# LOAD PRETRAINED UAV->S2A MODEL
# =========================================================

def load_pretrained_weights(model):

    print("\nLoading UAV pretrained model...")

    pretrained = torch.load(
        PRETRAIN_PATH,
        map_location=DEVICE
    )

    model_dict = model.state_dict()

    # =====================================================
    # REMOVE HEAD
    # =====================================================

    pretrained_dict = {

        k:v for k,v in pretrained.items()

        if (
            k in model_dict
            and "head" not in k
        )
    }

    model_dict.update(
        pretrained_dict
    )

    model.load_state_dict(
        model_dict
    )

    print(
        f"Loaded "
        f"{len(pretrained_dict)} "
        f"parameters"
    )

# =========================================================
# EVALUATE
# =========================================================

def evaluate(
    model,
    loader
):

    model.eval()

    y_true = []

    y_pred = []

    with torch.no_grad():

        for batch in loader:

            pred = model(

                batch["x"].to(DEVICE),

                batch["lambda"].to(DEVICE),

                batch["delta"].to(DEVICE),

                batch["type"].to(DEVICE),

                batch["temp"].to(DEVICE)
            )

            y_true.extend(
                batch["y"].cpu().numpy()
            )

            y_pred.extend(
                pred.cpu().numpy()
            )

    y_true = np.array(y_true)

    y_pred = np.array(y_pred)

    mse = mean_squared_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return mse, rmse, mae, r2

def train():

    print("\n========================")
    print("Sentinel-2 Transfer")
    print("========================")

    # =====================================================
    # dataset
    # =====================================================

    dataset = SentinelDataset(S2_PATH)

    # =====================================================
    # folds
    # =====================================================

    import json

    with open(SPLIT_PATH, "r") as f:

        folds = json.load(f)

    results = []

    # =====================================================
    # 5-fold
    # =====================================================

    for fold_id in range(5):

        print(
            f"\n================ "
            f"FOLD {fold_id}"
            f" ================"
        )

        # -------------------------------------------------
        # split
        # -------------------------------------------------

        train_idx = np.array(
            folds[fold_id]["train_idx"]
        )

        test_idx = np.array(
            folds[fold_id]["test_idx"]
        )

        train_ds = torch.utils.data.Subset(
            dataset,
            train_idx
        )

        test_ds = torch.utils.data.Subset(
            dataset,
            test_idx
        )

        train_loader = DataLoader(

            train_ds,

            batch_size=32,

            shuffle=True
        )

        test_loader = DataLoader(

            test_ds,

            batch_size=32,

            shuffle=False
        )

        # =================================================
        # model
        # =================================================

        model = (
            SpectralFusionModel()
            .to(DEVICE)
        )

        # =================================================
        # load pretrained
        # =================================================

        load_pretrained_weights(model)

        criterion = nn.MSELoss()

        # =================================================
        # STAGE1
        # =================================================

        print("\nStage1: adapter training")

        # -------------------------------------------------
        # freeze all
        # -------------------------------------------------

        for p in model.parameters():

            p.requires_grad = False

        # -------------------------------------------------
        # unfreeze input_embed
        # -------------------------------------------------

        for p in model.input_embed.parameters():

            p.requires_grad = True

        # -------------------------------------------------
        # unfreeze temp_embed
        # -------------------------------------------------

        for p in model.temp_embed.parameters():

            p.requires_grad = True

        # -------------------------------------------------
        # unfreeze FiLM
        # -------------------------------------------------

        for p in model.film.parameters():

            p.requires_grad = True

        # 解冻 weight2 相关网络
        for p in model.spectral_weight_net.parameters():
            
            p.requires_grad = True

        # -------------------------------------------------
        # unfreeze head
        # -------------------------------------------------

        for p in model.head.parameters():

            p.requires_grad = True

        optimizer = torch.optim.Adam(

            filter(
                lambda p: p.requires_grad,
                model.parameters()
            ),

            lr=1e-3
        )

        best_r2 = -999

        best_stage1 = None

        wait = 0

        patience = 50

        # =================================================
        # stage1 training
        # =================================================

        for epoch in range(500):

            model.train()

            total_loss = 0

            for batch in train_loader:

                pred = model(

                    batch["x"].to(DEVICE),

                    batch["lambda"].to(DEVICE),

                    batch["delta"].to(DEVICE),

                    batch["type"].to(DEVICE),

                    batch["temp"].to(DEVICE)
                )

                loss = criterion(

                    pred,

                    batch["y"].to(DEVICE)
                )

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()

                total_loss += loss.item()

            mse, rmse, mae, r2 = evaluate(
                model,
                test_loader
            )

            print(

                f"[Stage1] "

                f"Epoch {epoch:03d} | "

                f"Loss {total_loss:.4f} | "

                f"RMSE {rmse:.4f} | "

                f"R2 {r2:.4f}"
            )

            # -------------------------------------------------
            # early stopping
            # -------------------------------------------------

            if r2 > best_r2:

                best_r2 = r2

                best_stage1 = copy.deepcopy(
                    model.state_dict()
                )

                wait = 0

            else:

                wait += 1

                if wait >= patience:

                    print(
                        "\nStage1 Early stopping"
                    )

                    break

        # =================================================
        # restore best stage1
        # =================================================

        model.load_state_dict(
            best_stage1
        )

        # =================================================
        # STAGE2
        # =================================================

        print("\nStage2: partial finetuning")

        # -------------------------------------------------
        # unfreeze all
        # -------------------------------------------------

        for p in model.parameters():

            p.requires_grad = True

        # # -------------------------------------------------
        # # train adapter
        # # -------------------------------------------------

        # for p in model.input_embed.parameters():

        #     p.requires_grad = True

        # for p in model.film.parameters():

        #     p.requires_grad = True

        # for p in model.head.parameters():

        #     p.requires_grad = True

        # # -------------------------------------------------
        # # unfreeze fusion
        # # -------------------------------------------------

        # for p in model.fusion.parameters():

        #     p.requires_grad = True

        # # -------------------------------------------------
        # # unfreeze encoder last layer
        # # -------------------------------------------------

        # for p in model.encoder.layers[1].parameters():

        #     p.requires_grad = True

        optimizer = torch.optim.Adam(

            filter(
                lambda p: p.requires_grad,
                model.parameters()
            ),

            lr=1e-4
        )

        best_r2 = -999

        best_model = None

        wait = 0

        patience = 50

        # =================================================
        # stage2 training
        # =================================================

        for epoch in range(500):

            model.train()

            total_loss = 0

            for batch in train_loader:

                pred = model(

                    batch["x"].to(DEVICE),

                    batch["lambda"].to(DEVICE),

                    batch["delta"].to(DEVICE),

                    batch["type"].to(DEVICE),

                    batch["temp"].to(DEVICE)
                )

                loss = criterion(

                    pred,

                    batch["y"].to(DEVICE)
                )

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()

                total_loss += loss.item()

            mse, rmse, mae, r2 = evaluate(
                model,
                test_loader
            )

            print(

                f"[Stage2] "

                f"Epoch {epoch:03d} | "

                f"Loss {total_loss:.4f} | "

                f"RMSE {rmse:.4f} | "

                f"R2 {r2:.4f}"
            )

            # -------------------------------------------------
            # early stopping
            # -------------------------------------------------

            if r2 > best_r2:

                best_r2 = r2

                best_model = copy.deepcopy(
                    model.state_dict()
                )

                wait = 0

                torch.save(

                    best_model,

                    os.path.join(

                        SAVE_DIR,

                        f"best_fold_{fold_id}.pth"
                    )
                )

            else:

                wait += 1

                if wait >= patience:

                    print(
                        "\nStage2 Early stopping"
                    )

                    break

        # =================================================
        # final evaluation
        # =================================================

        model.load_state_dict(
            best_model
        )

        mse, rmse, mae, r2 = evaluate(
            model,
            test_loader
        )

        print("\n========================")
        print(f"FOLD {fold_id} RESULT")
        print("========================")

        print(f"MSE  : {mse:.4f}")
        print(f"RMSE : {rmse:.4f}")
        print(f"MAE  : {mae:.4f}")
        print(f"R2   : {r2:.4f}")

        results.append([

            mse,

            rmse,

            mae,

            r2
        ])

    # =====================================================
    # summary
    # =====================================================

    results = np.array(results)

    print("\n========================")
    print("5-FOLD FINAL RESULT")
    print("========================")

    print(

        f"MSE  : "
        f"{results[:,0].mean():.4f} ± "
        f"{results[:,0].std():.4f}"
    )

    print(

        f"RMSE : "
        f"{results[:,1].mean():.4f} ± "
        f"{results[:,1].std():.4f}"
    )

    print(

        f"MAE  : "
        f"{results[:,2].mean():.4f} ± "
        f"{results[:,2].std():.4f}"
    )

    print(

        f"R2   : "
        f"{results[:,3].mean():.4f} ± "
        f"{results[:,3].std():.4f}"
    )

# =========================================================

if __name__ == "__main__":

    train()


Sentinel-2 Transfer

================ FOLD 0 ================

Loading UAV->S2A pretrained model...
Loaded 50 parameters

Stage1: adapter training


C:\Users\PC\AppData\Local\Temp\ipykernel_22304\2040705861.py:480: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load(


[Stage1] Epoch 000 | Loss 1202.2618 | RMSE 2.6487 | R2 -1.2224
[Stage1] Epoch 001 | Loss 122.4189 | RMSE 1.7107 | R2 0.0729
[Stage1] Epoch 002 | Loss 86.7970 | RMSE 1.6435 | R2 0.1444
[Stage1] Epoch 003 | Loss 82.4345 | RMSE 1.6752 | R2 0.1111
[Stage1] Epoch 004 | Loss 77.8034 | RMSE 1.5989 | R2 0.1902
[Stage1] Epoch 005 | Loss 80.4424 | RMSE 1.6369 | R2 0.1512
[Stage1] Epoch 006 | Loss 75.4771 | RMSE 1.6723 | R2 0.1142
[Stage1] Epoch 007 | Loss 76.7825 | RMSE 1.6317 | R2 0.1566
[Stage1] Epoch 008 | Loss 75.0461 | RMSE 1.6212 | R2 0.1674
[Stage1] Epoch 009 | Loss 75.8109 | RMSE 1.6593 | R2 0.1279
[Stage1] Epoch 010 | Loss 73.9945 | RMSE 1.6564 | R2 0.1309
[Stage1] Epoch 011 | Loss 72.0771 | RMSE 1.5890 | R2 0.2002
[Stage1] Epoch 012 | Loss 71.7289 | RMSE 1.5691 | R2 0.2201
[Stage1] Epoch 013 | Loss 72.0755 | RMSE 1.5280 | R2 0.2604
[Stage1] Epoch 014 | Loss 72.9773 | RMSE 1.5052 | R2 0.2824
[Stage1] Epoch 015 | Loss 70.7864 | RMSE 1.5140 | R2 0.2739
[Stage1] Epoch 016 | Loss 69.6753 | 

C:\Users\PC\AppData\Local\Temp\ipykernel_22304\2040705861.py:480: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load(


[Stage1] Epoch 000 | Loss 1589.4241 | RMSE 4.8774 | R2 -6.8588
[Stage1] Epoch 001 | Loss 263.7374 | RMSE 2.0747 | R2 -0.4219
[Stage1] Epoch 002 | Loss 88.2889 | RMSE 1.7152 | R2 0.0282
[Stage1] Epoch 003 | Loss 79.5038 | RMSE 1.6937 | R2 0.0523
[Stage1] Epoch 004 | Loss 79.6884 | RMSE 1.6562 | R2 0.0939
[Stage1] Epoch 005 | Loss 76.9259 | RMSE 1.6588 | R2 0.0910
[Stage1] Epoch 006 | Loss 76.2983 | RMSE 1.6531 | R2 0.0972
[Stage1] Epoch 007 | Loss 75.5004 | RMSE 1.6134 | R2 0.1401
[Stage1] Epoch 008 | Loss 79.3322 | RMSE 1.6076 | R2 0.1462
[Stage1] Epoch 009 | Loss 73.9700 | RMSE 1.6540 | R2 0.0963
[Stage1] Epoch 010 | Loss 73.6246 | RMSE 1.6362 | R2 0.1156
[Stage1] Epoch 011 | Loss 70.9511 | RMSE 1.5934 | R2 0.1613
[Stage1] Epoch 012 | Loss 69.4572 | RMSE 1.6537 | R2 0.0966
[Stage1] Epoch 013 | Loss 71.4480 | RMSE 1.6575 | R2 0.0924
[Stage1] Epoch 014 | Loss 69.8435 | RMSE 1.5910 | R2 0.1637
[Stage1] Epoch 015 | Loss 70.7551 | RMSE 1.5726 | R2 0.1830
[Stage1] Epoch 016 | Loss 68.4697 |

...

[Stage1] Epoch 001 | Loss 161.3259 | RMSE 1.9282 | R2 -0.3629
[Stage1] Epoch 002 | Loss 96.6591 | RMSE 1.5879 | R2 0.0757
[Stage1] Epoch 003 | Loss 83.2756 | RMSE 1.5914 | R2 0.0717
[Stage1] Epoch 004 | Loss 79.0383 | RMSE 1.6044 | R2 0.0564
[Stage1] Epoch 005 | Loss 78.6570 | RMSE 1.5241 | R2 0.1485
[Stage1] Epoch 006 | Loss 76.4776 | RMSE 1.5280 | R2 0.1441
[Stage1] Epoch 007 | Loss 77.3496 | RMSE 1.5334 | R2 0.1380
[Stage1] Epoch 008 | Loss 74.1386 | RMSE 1.5122 | R2 0.1618
[Stage1] Epoch 009 | Loss 74.8905 | RMSE 1.4877 | R2 0.1887
[Stage1] Epoch 010 | Loss 74.0581 | RMSE 1.5211 | R2 0.1518
[Stage1] Epoch 011 | Loss 74.0150 | RMSE 1.5058 | R2 0.1688
[Stage1] Epoch 012 | Loss 71.1438 | RMSE 1.5037 | R2 0.1712
[Stage1] Epoch 013 | Loss 71.2372 | RMSE 1.4900 | R2 0.1862
[Stage1] Epoch 014 | Loss 70.4515 | RMSE 1.4964 | R2 0.1792
[Stage1] Epoch 015 | Loss 71.1530 | RMSE 1.5001 | R2 0.1750
[Stage1] Epoch 016 | Loss 69.3854 | RMSE 1.4970 | R2 0.1785
[Stage1] Epoch 017 | Loss 69.7180 | RM

C:\Users\PC\AppData\Local\Temp\ipykernel_22304\2040705861.py:480: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load(


[Stage1] Epoch 001 | Loss 151.8332 | RMSE 1.9810 | R2 -0.2865
[Stage1] Epoch 002 | Loss 88.5878 | RMSE 1.6727 | R2 0.0829
[Stage1] Epoch 003 | Loss 81.9782 | RMSE 1.6522 | R2 0.1051
[Stage1] Epoch 004 | Loss 79.6107 | RMSE 1.6825 | R2 0.0720
[Stage1] Epoch 005 | Loss 78.1041 | RMSE 1.6231 | R2 0.1364
[Stage1] Epoch 006 | Loss 76.4341 | RMSE 1.6549 | R2 0.1023
[Stage1] Epoch 007 | Loss 74.4881 | RMSE 1.6530 | R2 0.1043
[Stage1] Epoch 008 | Loss 76.1352 | RMSE 1.6254 | R2 0.1339
[Stage1] Epoch 009 | Loss 75.5283 | RMSE 1.6326 | R2 0.1263
[Stage1] Epoch 010 | Loss 74.6149 | RMSE 1.5967 | R2 0.1643
[Stage1] Epoch 011 | Loss 73.8295 | RMSE 1.5740 | R2 0.1879
[Stage1] Epoch 012 | Loss 72.1828 | RMSE 1.6711 | R2 0.0845
[Stage1] Epoch 013 | Loss 71.5134 | RMSE 1.6441 | R2 0.1139
[Stage1] Epoch 014 | Loss 71.2221 | RMSE 1.5995 | R2 0.1613
[Stage1] Epoch 015 | Loss 70.6911 | RMSE 1.5511 | R2 0.2113
[Stage1] Epoch 016 | Loss 70.3822 | RMSE 1.5479 | R2 0.2146
[Stage1] Epoch 017 | Loss 68.4690 | RM

C:\Users\PC\AppData\Local\Temp\ipykernel_22304\2040705861.py:480: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load(


[Stage1] Epoch 001 | Loss 253.1498 | RMSE 2.0303 | R2 -0.5033
[Stage1] Epoch 002 | Loss 92.1191 | RMSE 1.6322 | R2 0.0285
[Stage1] Epoch 003 | Loss 83.2462 | RMSE 1.6549 | R2 0.0013
[Stage1] Epoch 004 | Loss 83.0490 | RMSE 1.6180 | R2 0.0452
[Stage1] Epoch 005 | Loss 79.5033 | RMSE 1.5843 | R2 0.0846
[Stage1] Epoch 006 | Loss 79.6822 | RMSE 1.5655 | R2 0.1062
[Stage1] Epoch 007 | Loss 79.9641 | RMSE 1.5288 | R2 0.1477
[Stage1] Epoch 008 | Loss 78.4832 | RMSE 1.5862 | R2 0.0824
[Stage1] Epoch 009 | Loss 75.8205 | RMSE 1.5650 | R2 0.1068
[Stage1] Epoch 010 | Loss 76.6805 | RMSE 1.5341 | R2 0.1417
[Stage1] Epoch 011 | Loss 76.2130 | RMSE 1.5876 | R2 0.0808
[Stage1] Epoch 012 | Loss 75.0126 | RMSE 1.5892 | R2 0.0790
[Stage1] Epoch 013 | Loss 75.7066 | RMSE 1.5882 | R2 0.0801
[Stage1] Epoch 014 | Loss 73.2921 | RMSE 1.5178 | R2 0.1599
[Stage1] Epoch 015 | Loss 74.1194 | RMSE 1.5351 | R2 0.1406
[Stage1] Epoch 016 | Loss 73.5359 | RMSE 1.5624 | R2 0.1098
[Stage1] Epoch 017 | Loss 73.5779 | RM

### Landsat 8

## UAV pretrain transfer to Landsat 8 

In [ ]:
import os
import copy
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================================================
# PATH
# =========================================================
DATA_DIR = r"\data\input"

LANDSAT_PATH = os.path.join(DATA_DIR,"Landsat-8.csv")

# UAV pretrained full model
PRETRAIN_PATH = (
    r"\data\input"
    r"\UAV\explain\full_model_explain.pth"
)

SAVE_DIR = os.path.join(
    DATA_DIR,
    "Landsat8_transfer"
)

os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# =========================================================
# GLOBAL WAVELENGTH NORMALIZATION
# =========================================================
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002.0
def normalize_wavelength(x):
    return ((x - GLOBAL_LAMBDA_MIN)/ (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN))

# =========================================================
# DATASET
# =========================================================

class LandsatDataset(Dataset):
    def __init__(self, csv_path):
        df = pd.read_csv(
            csv_path,
            encoding="gbk"
        )
        # -------------------------------------------------
        # Landsat-8 bands
        # -------------------------------------------------
        self.band_names = [
            "SR_B1",
            "SR_B2",
            "SR_B3",
            "SR_B4",
            "SR_B5"
        ]
        # -------------------------------------------------
        # center wavelength
        # -------------------------------------------------
        self.lambda_ = np.array([
            443,
            482,
            561,
            655,
            865
        ], dtype=np.float32)
        # -------------------------------------------------
        # bandwidth
        # -------------------------------------------------
        self.delta = np.array([
            20,
            65,
            75,
            50,
            40
        ], dtype=np.float32)
        # -------------------------------------------------
        # input
        # -------------------------------------------------

        self.x = df[
            self.band_names
        ].values.astype(np.float32)

        # -------------------------------------------------
        # target
        # -------------------------------------------------

        self.y = df.iloc[:, -1].values.astype(np.float32)

        # -------------------------------------------------
        # water temperature
        # -------------------------------------------------

        temp = df.iloc[:, -2].values.astype(np.float32)

        # -------------------------------------------------
        # NDWI filter
        # -------------------------------------------------

        ndwi = (
            df["SR_B3"] - df["SR_B5"]
        ) / (
            df["SR_B3"] + df["SR_B5"] + 1e-8
        )

        mask = ndwi > 0.1

        self.x = self.x[mask]
        self.y = self.y[mask]
        temp = temp[mask]

        # -------------------------------------------------
        # remove invalid
        # -------------------------------------------------

        mask = np.all(self.x >= 0, axis=1)

        self.x = self.x[mask]
        self.y = self.y[mask]
        temp = temp[mask]

        # -------------------------------------------------
        # normalization
        # -------------------------------------------------

        self.x_mean = self.x.mean(axis=0)
        self.x_std = self.x.std(axis=0) + 1e-6

        self.x = (
            self.x - self.x_mean
        ) / self.x_std

        self.temp = (
            temp - temp.mean()
        ) / (
            temp.std() + 1e-6
        )

        self.lambda_ = normalize_wavelength(
            self.lambda_
        )

        self.delta = normalize_wavelength(
            self.delta
        )

    def __len__(self):

        return len(self.x)

    def __getitem__(self, idx):

        return {
            "x": torch.tensor(
                self.x[idx]
            ),

            "lambda": torch.tensor(
                self.lambda_
            ),

            "delta": torch.tensor(
                self.delta
            ),

            "temp": torch.tensor(
                self.temp[idx]
            ),

            # Landsat dtype = 2
            "type": torch.tensor(2),

            "y": torch.tensor(
                self.y[idx]
            )
        }

# =========================================================
# SENSOR-SPECIFIC ADAPTER
# =========================================================

class LandsatInputAdapter(nn.Module):

    def __init__(self, d_model):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(3, d_model),

            nn.ReLU(),

            nn.Linear(
                d_model,
                d_model
            )
        )

    def forward(self, x):

        return self.net(x)

# =========================================================
# SENSOR-SPECIFIC FILM
# =========================================================

class LandsatFiLM(nn.Module):

    def __init__(self, d_model):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(4, 128),

            nn.SiLU(),

            nn.Linear(128, 128),

            nn.SiLU(),

            nn.Linear(
                128,
                d_model * 2
            )
        )

    def forward(self, cond):

        gamma, beta = (
            self.net(cond)
            .chunk(2, dim=1)
        )

        return gamma, beta

# =========================================================
# SHARED BACKBONE MODEL
# =========================================================

class LandsatTransferModel(nn.Module):

    def __init__(
        self,
        d_model=64,
        grid_size=16
    ):

        super().__init__()

        self.d_model = d_model

        # =================================================
        # spectral grid
        # =================================================

        self.register_buffer(
            "grid",
            torch.linspace(
                0,
                1,
                grid_size
            )
        )

        # =================================================
        # sensor-specific adapter
        # =================================================

        self.input_embed = (
            LandsatInputAdapter(d_model)
        )

        # =================================================
        # sensor-specific film
        # =================================================

        self.film = (
            LandsatFiLM(d_model)
        )

        # =================================================
        # shared Fourier
        # =================================================

        self.fourier_B = nn.Parameter(
            torch.logspace(
                0,
                2,
                d_model // 2
            )
        )

        self.fourier_scale = nn.Parameter(
            torch.ones(1)
        )

        # =================================================
        # shared fusion
        # =================================================

        self.fusion = nn.Linear(
            d_model * 2,
            d_model
        )

        # =================================================
        # shared transformer
        # =================================================

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=4,
                batch_first=True
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=2
            )
        )

        # =================================================
        # shared CLS token
        # =================================================

        self.cls_token = nn.Parameter(
            torch.randn(
                1,
                1,
                d_model
            )
        )

        # =================================================
        # new Landsat head
        # =================================================

        self.head = nn.Sequential(

            nn.Linear(
                d_model,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                1
            )
        )

    # =====================================================
    # Fourier encoding
    # =====================================================

    def fourier_encoding(self, x):

        x = x.unsqueeze(-1)

        proj = (
            x
            * self.fourier_B
            * self.fourier_scale
        )

        return torch.cat([

            torch.sin(proj),

            torch.cos(proj)

        ], dim=-1)

    # =====================================================
    # spectral projection
    # =====================================================

    def spectral_project(
        self,
        z,
        wl,
        delta
    ):

        grid = (
            self.grid
            .to(z.device)
            .unsqueeze(0)
            .unsqueeze(0)
        )

        wl = wl.unsqueeze(-1)

        delta = delta.unsqueeze(-1)

        sigma = delta + 1e-6

        weight = torch.exp(
            -((wl - grid) ** 2)
            / (sigma ** 2)
        )

        weight = (
            weight
            / (
                weight.sum(
                    dim=1,
                    keepdim=True
                ) + 1e-8
            )
        )

        return (
            z.unsqueeze(2)
            * weight.unsqueeze(-1)
        ).sum(dim=1)

    # =====================================================
    # forward
    # =====================================================

    def forward(
        self,
        x,
        wl,
        delta,
        dtype,
        temp
    ):

        # -------------------------------------------------
        # input
        # -------------------------------------------------

        x = x.unsqueeze(-1)

        wl = wl.unsqueeze(-1)

        delta = delta.unsqueeze(-1)

        inp = torch.cat([
            x,
            wl,
            delta
        ], dim=-1)

        # -------------------------------------------------
        # Landsat adapter
        # -------------------------------------------------

        z = self.input_embed(inp)

        # -------------------------------------------------
        # Landsat FiLM
        # -------------------------------------------------

        cond = torch.stack([

            dtype.float(),

            temp,

            wl.mean(dim=1).squeeze(-1),

            delta.mean(dim=1).squeeze(-1)

        ], dim=1)

        gamma, beta = self.film(cond)

        z = (
            gamma.unsqueeze(1) * z
            + beta.unsqueeze(1)
        )

        # -------------------------------------------------
        # shared spectral projection
        # -------------------------------------------------

        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1)
        )

        # -------------------------------------------------
        # shared Fourier encoding
        # -------------------------------------------------

        grid = (
            self.grid
            .unsqueeze(0)
            .expand(z.size(0), -1)
        )

        pos = self.fourier_encoding(grid)

        z = torch.cat([
            z,
            pos
        ], dim=-1)

        # -------------------------------------------------
        # shared fusion
        # -------------------------------------------------

        z = self.fusion(z)

        # -------------------------------------------------
        # shared transformer
        # -------------------------------------------------

        B = z.size(0)

        cls = self.cls_token.expand(
            B,
            -1,
            -1
        )

        z = torch.cat([
            cls,
            z
        ], dim=1)

        z = self.encoder(z)

        z = z[:, 0]

        # -------------------------------------------------
        # Landsat head
        # -------------------------------------------------

        return self.head(z).squeeze()

# =========================================================
# LOAD UAV SHARED BACKBONE
# =========================================================

def load_uav_shared_weights(model):

    print("\nLoading UAV shared backbone...")

    pretrained = torch.load(
        PRETRAIN_PATH,
        map_location=DEVICE
    )

    model_dict = model.state_dict()

    # =====================================================
    # ONLY LOAD SHARED MODULES
    # =====================================================

    shared_keys = [

        "fourier_B",

        "fourier_scale",

        "fusion.weight",
        "fusion.bias",

        "encoder.layers.0.self_attn.in_proj_weight",
        "encoder.layers.0.self_attn.in_proj_bias",
        "encoder.layers.0.self_attn.out_proj.weight",
        "encoder.layers.0.self_attn.out_proj.bias",

        "encoder.layers.0.linear1.weight",
        "encoder.layers.0.linear1.bias",

        "encoder.layers.0.linear2.weight",
        "encoder.layers.0.linear2.bias",

        "encoder.layers.0.norm1.weight",
        "encoder.layers.0.norm1.bias",

        "encoder.layers.0.norm2.weight",
        "encoder.layers.0.norm2.bias",

        "encoder.layers.1.self_attn.in_proj_weight",
        "encoder.layers.1.self_attn.in_proj_bias",
        "encoder.layers.1.self_attn.out_proj.weight",
        "encoder.layers.1.self_attn.out_proj.bias",

        "encoder.layers.1.linear1.weight",
        "encoder.layers.1.linear1.bias",

        "encoder.layers.1.linear2.weight",
        "encoder.layers.1.linear2.bias",

        "encoder.layers.1.norm1.weight",
        "encoder.layers.1.norm1.bias",

        "encoder.layers.1.norm2.weight",
        "encoder.layers.1.norm2.bias",

        "cls_token"
    ]

    loaded = 0

    for k in shared_keys:

        if (
            k in pretrained
            and k in model_dict
        ):

            model_dict[k] = pretrained[k]

            loaded += 1

    model.load_state_dict(model_dict)

    print(
        f"Loaded {loaded} shared parameters"
    )

# =========================================================
# EVALUATE
# =========================================================

def evaluate(
    model,
    loader
):

    model.eval()

    y_true = []

    y_pred = []

    with torch.no_grad():

        for batch in loader:

            pred = model(

                batch["x"].to(DEVICE),

                batch["lambda"].to(DEVICE),

                batch["delta"].to(DEVICE),

                batch["type"].to(DEVICE),

                batch["temp"].to(DEVICE)
            )

            y_true.extend(
                batch["y"].cpu().numpy()
            )

            y_pred.extend(
                pred.cpu().numpy()
            )

    y_true = np.array(y_true)

    y_pred = np.array(y_pred)

    mse = mean_squared_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return mse, rmse, mae, r2

# =========================================================
# TRAIN
# =========================================================

def train():

    print("\n========================")
    print("LANDSAT-8 TRANSFER")
    print("========================")

    dataset = LandsatDataset(
        LANDSAT_PATH
    )

    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=True
    )

    model = (
        LandsatTransferModel()
        .to(DEVICE)
    )

    # =====================================================
    # load UAV shared backbone
    # =====================================================

    load_uav_shared_weights(model)

    # =====================================================
    # freeze shared backbone
    # =====================================================

    print("\nStage1: train Landsat adapters")

    for name, p in model.named_parameters():

        if (
            "input_embed" not in name
            and "film" not in name
            and "head" not in name
        ):
            p.requires_grad = False

    optimizer = torch.optim.Adam(

        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),

        lr=1e-3
    )

    criterion = nn.MSELoss()

    best_r2 = -999

    best_model = None

    # =====================================================
    # stage1
    # =====================================================

    for epoch in range(50):

        model.train()

        total_loss = 0

        for batch in loader:

            pred = model(

                batch["x"].to(DEVICE),

                batch["lambda"].to(DEVICE),

                batch["delta"].to(DEVICE),

                batch["type"].to(DEVICE),

                batch["temp"].to(DEVICE)
            )

            loss = criterion(
                pred,
                batch["y"].to(DEVICE)
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        mse, rmse, mae, r2 = evaluate(
            model,
            loader
        )

        print(
            f"[Stage1] "
            f"Epoch {epoch:03d} | "
            f"Loss {total_loss:.4f} | "
            f"R2 {r2:.4f}"
        )

    # =====================================================
    # stage2
    # =====================================================

    print("\nStage2: finetune all")

    for p in model.parameters():

        p.requires_grad = True

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-4
    )

    patience = 30

    wait = 0

    # =====================================================
    # finetune
    # =====================================================

    for epoch in range(200):

        model.train()

        total_loss = 0

        for batch in loader:

            pred = model(

                batch["x"].to(DEVICE),

                batch["lambda"].to(DEVICE),

                batch["delta"].to(DEVICE),

                batch["type"].to(DEVICE),

                batch["temp"].to(DEVICE)
            )

            loss = criterion(
                pred,
                batch["y"].to(DEVICE)
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        mse, rmse, mae, r2 = evaluate(
            model,
            loader
        )

        print(
            f"[Stage2] "
            f"Epoch {epoch:03d} | "
            f"Loss {total_loss:.4f} | "
            f"RMSE {rmse:.4f} | "
            f"R2 {r2:.4f}"
        )

        # -------------------------------------------------
        # save best
        # -------------------------------------------------

        if r2 > best_r2:

            best_r2 = r2

            best_model = copy.deepcopy(
                model.state_dict()
            )

            wait = 0

            torch.save(

                best_model,

                os.path.join(
                    SAVE_DIR,
                    "best_landsat_transfer.pth"
                )
            )

        else:

            wait += 1

            if wait >= patience:

                print(
                    "\nEarly stopping"
                )

                break

    # =====================================================
    # final evaluation
    # =====================================================

    model.load_state_dict(
        best_model
    )

    mse, rmse, mae, r2 = evaluate(
        model,
        loader
    )

    print("\n========================")
    print("FINAL RESULT")
    print("========================")

    print(f"MSE  : {mse:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAE  : {mae:.4f}")
    print(f"R2   : {r2:.4f}")

# =========================================================

if __name__ == "__main__":

    train()


LANDSAT-8 TRANSFER

Loading UAV shared backbone...


C:\Users\PC\AppData\Local\Temp\ipykernel_47308\2968340953.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load(


Loaded 29 shared parameters

Stage1: train Landsat adapters
[Stage1] Epoch 000 | Loss 872.7293 | R2 -6.1871
[Stage1] Epoch 001 | Loss 204.8687 | R2 -0.0226
[Stage1] Epoch 002 | Loss 59.1456 | R2 0.0573
[Stage1] Epoch 003 | Loss 52.9526 | R2 0.2364
[Stage1] Epoch 004 | Loss 52.9793 | R2 0.2253
[Stage1] Epoch 005 | Loss 50.4464 | R2 0.2103
[Stage1] Epoch 006 | Loss 49.5070 | R2 0.2154
[Stage1] Epoch 007 | Loss 49.6424 | R2 0.1912
[Stage1] Epoch 008 | Loss 49.9322 | R2 0.2296
[Stage1] Epoch 009 | Loss 49.3688 | R2 0.1523
[Stage1] Epoch 010 | Loss 46.9562 | R2 0.1366
[Stage1] Epoch 011 | Loss 51.3570 | R2 0.0797
[Stage1] Epoch 012 | Loss 49.4030 | R2 0.1929
[Stage1] Epoch 013 | Loss 49.7233 | R2 0.2486
[Stage1] Epoch 014 | Loss 48.9591 | R2 0.2327
[Stage1] Epoch 015 | Loss 48.7176 | R2 0.2541
[Stage1] Epoch 016 | Loss 47.3720 | R2 0.1834
[Stage1] Epoch 017 | Loss 49.0120 | R2 0.1619
[Stage1] Epoch 018 | Loss 47.6426 | R2 0.2163
[Stage1] Epoch 019 | Loss 45.9389 | R2 0.2213
[Stage1] Epoch 0

## Create GroupKFold splits and save them

In [ ]:
import numpy as np
import json
import os
import pandas as pd

def build_group_kfold_split(groups, n_splits=5, seed=42):
    np.random.seed(seed)

    unique_groups = np.unique(groups)
    np.random.shuffle(unique_groups)

    fold_size = len(unique_groups) // n_splits
    folds = []

    for i in range(n_splits):
        test_groups = unique_groups[i * fold_size:(i + 1) * fold_size]
        train_groups = np.setdiff1d(unique_groups, test_groups)

        train_idx = np.where(np.isin(groups, train_groups))[0].tolist()
        test_idx = np.where(np.isin(groups, test_groups))[0].tolist()

        folds.append({
            "train_idx": train_idx,
            "test_idx": test_idx
        })

    return folds


csv_path = r"data/input/Landsat-8.csv"
df = pd.read_csv(csv_path, encoding="gbk")


# NDWI
ndwi = (df['SR_B3'] - df['SR_B5']) / (df['SR_B3'] + df['SR_B5'] + 1e-8)
df = df[ndwi > 0.1]


x = df.iloc[:, :-2].values
mask = (x >= 0).all(axis=1)
df = df[mask]


df = df.reset_index(drop=True)

print("Filtered size:", len(df))



groups = np.arange(len(df)) 

folds = build_group_kfold_split(groups, n_splits=5, seed=24)

# save
save_path = r"data/input/Landsat-8/Landsat-8_splits.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

with open(save_path, "w") as f:
    json.dump(folds, f, indent=2)

print("Saved split to:", save_path)

Filtered size: 640
Saved split to: data/input/Landsat-8/Landsat-8_splits.json


## model train

In [ ]:
import os
import copy
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import json
from torch.utils.data import Subset

# =========================================================
# LOAD SPLIT
# =========================================================

SPLIT_PATH = os.path.join(
    DATA_DIR,
    "Landsat-8/Landsat-8_splits.json"
)

def load_split(path):

    with open(path, "r") as f:

        return json.load(f)
# =========================================================
# PATH
# =========================================================
DATA_DIR = r"\data\input"

LANDSAT_PATH = os.path.join(DATA_DIR,"Landsat-8.csv")

# UAV pretrained full model
PRETRAIN_PATH = (
    r"\data\input"
    r"\UAV\explain\full_model_explain.pth"
)

SAVE_DIR = os.path.join(
    DATA_DIR,
    "Landsat8_transfer"
)

os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# =========================================================
# GLOBAL WAVELENGTH NORMALIZATION
# =========================================================
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002.0
def normalize_wavelength(x):
    return ((x - GLOBAL_LAMBDA_MIN)/ (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN))

# =========================================================
# DATASET
# =========================================================

class LandsatDataset(Dataset):

    def __init__(self, csv_path):

        df = pd.read_csv(
            csv_path,
            encoding="gbk"
        )

        # -------------------------------------------------
        # Landsat-8 bands
        # -------------------------------------------------

        self.band_names = [
            "SR_B1",
            "SR_B2",
            "SR_B3",
            "SR_B4",
            "SR_B5"
        ]

        # -------------------------------------------------
        # center wavelength
        # -------------------------------------------------

        self.lambda_ = np.array([
            443,
            482,
            561,
            655,
            865
        ], dtype=np.float32)

        # -------------------------------------------------
        # bandwidth
        # -------------------------------------------------

        self.delta = np.array([
            20,
            65,
            75,
            50,
            40
        ], dtype=np.float32)

        # -------------------------------------------------
        # input
        # -------------------------------------------------

        self.x = df[
            self.band_names
        ].values.astype(np.float32)

        # -------------------------------------------------
        # target
        # -------------------------------------------------

        self.y = df.iloc[:, -1].values.astype(np.float32)

        # -------------------------------------------------
        # water temperature
        # -------------------------------------------------

        temp = df.iloc[:, -2].values.astype(np.float32)

        # -------------------------------------------------
        # NDWI filter
        # -------------------------------------------------

        ndwi = (
            df["SR_B3"] - df["SR_B5"]
        ) / (
            df["SR_B3"] + df["SR_B5"] + 1e-8
        )

        mask = ndwi > 0.1

        self.x = self.x[mask]
        self.y = self.y[mask]
        temp = temp[mask]

        # -------------------------------------------------
        # remove invalid
        # -------------------------------------------------

        mask = np.all(self.x >= 0, axis=1)

        self.x = self.x[mask]
        self.y = self.y[mask]
        temp = temp[mask]

        # -------------------------------------------------
        # normalization
        # -------------------------------------------------

        self.x_mean = self.x.mean(axis=0)
        self.x_std = self.x.std(axis=0) + 1e-6

        self.x = (
            self.x - self.x_mean
        ) / self.x_std

        self.temp = (
            temp - temp.mean()
        ) / (
            temp.std() + 1e-6
        )

        self.lambda_ = normalize_wavelength(
            self.lambda_
        )

        self.delta = normalize_wavelength(
            self.delta
        )

    def __len__(self):

        return len(self.x)

    def __getitem__(self, idx):

        return {
            "x": torch.tensor(
                self.x[idx]
            ),

            "lambda": torch.tensor(
                self.lambda_
            ),

            "delta": torch.tensor(
                self.delta
            ),

            "temp": torch.tensor(
                self.temp[idx]
            ),

            # Landsat dtype = 2
            "type": torch.tensor(2),

            "y": torch.tensor(
                self.y[idx]
            )
        }

# =========================================================
# SENSOR-SPECIFIC ADAPTER
# =========================================================

class LandsatInputAdapter(nn.Module):

    def __init__(self, d_model):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(3, d_model),

            nn.ReLU(),

            nn.Linear(
                d_model,
                d_model
            )
        )

    def forward(self, x):

        return self.net(x)

# =========================================================
# SENSOR-SPECIFIC FILM
# =========================================================

class LandsatFiLM(nn.Module):

    def __init__(self, d_model):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(4, 128),

            nn.SiLU(),

            nn.Linear(128, 128),

            nn.SiLU(),

            nn.Linear(
                128,
                d_model * 2
            )
        )

    def forward(self, cond):

        gamma, beta = (
            self.net(cond)
            .chunk(2, dim=1)
        )

        return gamma, beta

# =========================================================
# SHARED BACKBONE MODEL
# =========================================================

class LandsatTransferModel(nn.Module):

    def __init__(
        self,
        d_model=64,
        grid_size=16
    ):

        super().__init__()

        self.d_model = d_model

        # =================================================
        # spectral grid
        # =================================================

        self.register_buffer(
            "grid",
            torch.linspace(
                0,
                1,
                grid_size
            )
        )

        # =================================================
        # sensor-specific adapter
        # =================================================

        self.input_embed = (
            LandsatInputAdapter(d_model)
        )

        # =================================================
        # sensor-specific film
        # =================================================

        self.film = (
            LandsatFiLM(d_model)
        )

        # =================================================
        # shared Fourier
        # =================================================

        self.fourier_B = nn.Parameter(
            torch.logspace(
                0,
                2,
                d_model // 2
            )
        )

        self.fourier_scale = nn.Parameter(
            torch.ones(1)
        )

        # =================================================
        # shared fusion
        # =================================================

        self.fusion = nn.Linear(
            d_model * 2,
            d_model
        )

        # =================================================
        # shared transformer
        # =================================================

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=4,
                batch_first=True
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=2
            )
        )

        # =================================================
        # shared CLS token
        # =================================================

        self.cls_token = nn.Parameter(
            torch.randn(
                1,
                1,
                d_model
            )
        )

        # =================================================
        # new Landsat head
        # =================================================

        self.head = nn.Sequential(

            nn.Linear(
                d_model,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                1
            )
        )

    # =====================================================
    # Fourier encoding
    # =====================================================

    def fourier_encoding(self, x):

        x = x.unsqueeze(-1)

        proj = (
            x
            * self.fourier_B
            * self.fourier_scale
        )

        return torch.cat([

            torch.sin(proj),

            torch.cos(proj)

        ], dim=-1)

    # =====================================================
    # spectral projection
    # =====================================================

    def spectral_project(
        self,
        z,
        wl,
        delta
    ):

        grid = (
            self.grid
            .to(z.device)
            .unsqueeze(0)
            .unsqueeze(0)
        )

        wl = wl.unsqueeze(-1)

        delta = delta.unsqueeze(-1)

        sigma = delta + 1e-6

        weight = torch.exp(
            -((wl - grid) ** 2)
            / (sigma ** 2)
        )

        weight = (
            weight
            / (
                weight.sum(
                    dim=1,
                    keepdim=True
                ) + 1e-8
            )
        )

        return (
            z.unsqueeze(2)
            * weight.unsqueeze(-1)
        ).sum(dim=1)

    # =====================================================
    # forward
    # =====================================================

    def forward(
        self,
        x,
        wl,
        delta,
        dtype,
        temp
    ):

        # -------------------------------------------------
        # input
        # -------------------------------------------------

        x = x.unsqueeze(-1)

        wl = wl.unsqueeze(-1)

        delta = delta.unsqueeze(-1)

        inp = torch.cat([
            x,
            wl,
            delta
        ], dim=-1)

        # -------------------------------------------------
        # Landsat adapter
        # -------------------------------------------------

        z = self.input_embed(inp)

        # -------------------------------------------------
        # Landsat FiLM
        # -------------------------------------------------

        cond = torch.stack([

            dtype.float(),

            temp,

            wl.mean(dim=1).squeeze(-1),

            delta.mean(dim=1).squeeze(-1)

        ], dim=1)

        gamma, beta = self.film(cond)

        z = (
            gamma.unsqueeze(1) * z
            + beta.unsqueeze(1)
        )

        # -------------------------------------------------
        # shared spectral projection
        # -------------------------------------------------

        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1)
        )

        # -------------------------------------------------
        # shared Fourier encoding
        # -------------------------------------------------

        grid = (
            self.grid
            .unsqueeze(0)
            .expand(z.size(0), -1)
        )

        pos = self.fourier_encoding(grid)

        z = torch.cat([
            z,
            pos
        ], dim=-1)

        # -------------------------------------------------
        # shared fusion
        # -------------------------------------------------

        z = self.fusion(z)

        # -------------------------------------------------
        # shared transformer
        # -------------------------------------------------

        B = z.size(0)

        cls = self.cls_token.expand(
            B,
            -1,
            -1
        )

        z = torch.cat([
            cls,
            z
        ], dim=1)

        z = self.encoder(z)

        z = z[:, 0]

        # -------------------------------------------------
        # Landsat head
        # -------------------------------------------------

        return self.head(z).squeeze()

# =========================================================
# LOAD UAV SHARED BACKBONE
# =========================================================

def load_uav_shared_weights(model):

    print("\nLoading UAV shared backbone...")

    pretrained = torch.load(
        PRETRAIN_PATH,
        map_location=DEVICE
    )

    model_dict = model.state_dict()

    # =====================================================
    # ONLY LOAD SHARED MODULES
    # =====================================================

    shared_keys = [

        "fourier_B",

        "fourier_scale",

        "fusion.weight",
        "fusion.bias",

        "encoder.layers.0.self_attn.in_proj_weight",
        "encoder.layers.0.self_attn.in_proj_bias",
        "encoder.layers.0.self_attn.out_proj.weight",
        "encoder.layers.0.self_attn.out_proj.bias",

        "encoder.layers.0.linear1.weight",
        "encoder.layers.0.linear1.bias",

        "encoder.layers.0.linear2.weight",
        "encoder.layers.0.linear2.bias",

        "encoder.layers.0.norm1.weight",
        "encoder.layers.0.norm1.bias",

        "encoder.layers.0.norm2.weight",
        "encoder.layers.0.norm2.bias",

        "encoder.layers.1.self_attn.in_proj_weight",
        "encoder.layers.1.self_attn.in_proj_bias",
        "encoder.layers.1.self_attn.out_proj.weight",
        "encoder.layers.1.self_attn.out_proj.bias",

        "encoder.layers.1.linear1.weight",
        "encoder.layers.1.linear1.bias",

        "encoder.layers.1.linear2.weight",
        "encoder.layers.1.linear2.bias",

        "encoder.layers.1.norm1.weight",
        "encoder.layers.1.norm1.bias",

        "encoder.layers.1.norm2.weight",
        "encoder.layers.1.norm2.bias",

        "cls_token"
    ]

    loaded = 0

    for k in shared_keys:

        if (
            k in pretrained
            and k in model_dict
        ):

            model_dict[k] = pretrained[k]

            loaded += 1

    model.load_state_dict(model_dict)

    print(
        f"Loaded {loaded} shared parameters"
    )

# =========================================================
# EVALUATE
# =========================================================

def evaluate(
    model,
    loader
):

    model.eval()

    y_true = []

    y_pred = []

    with torch.no_grad():

        for batch in loader:

            pred = model(

                batch["x"].to(DEVICE),

                batch["lambda"].to(DEVICE),

                batch["delta"].to(DEVICE),

                batch["type"].to(DEVICE),

                batch["temp"].to(DEVICE)
            )

            y_true.extend(
                batch["y"].cpu().numpy()
            )

            y_pred.extend(
                pred.cpu().numpy()
            )

    y_true = np.array(y_true)

    y_pred = np.array(y_pred)

    mse = mean_squared_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return mse, rmse, mae, r2

# =========================================================
# TRAIN
# =========================================================

def train():

    print("\n========================")
    print("LANDSAT-8 TRANSFER")
    print("========================")

    # =====================================================
    # full dataset
    # =====================================================

    full_dataset = LandsatDataset(
        LANDSAT_PATH
    )

    # =====================================================
    # load folds
    # =====================================================

    folds = load_split(SPLIT_PATH)

    # =====================================================
    # save all fold result
    # =====================================================

    all_results = []

    # =====================================================
    # 5-fold CV
    # =====================================================

    for fold_id in range(5):

        print(
            f"\n================ "
            f"FOLD {fold_id}"
            f" ================"
        )

        # -------------------------------------------------
        # split
        # -------------------------------------------------

        train_idx = np.array(
            folds[fold_id]["train_idx"]
        )

        test_idx = np.array(
            folds[fold_id]["test_idx"]
        )

        train_dataset = Subset(
            full_dataset,
            train_idx
        )

        test_dataset = Subset(
            full_dataset,
            test_idx
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=32,
            shuffle=True
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=32,
            shuffle=False
        )

        # =================================================
        # model
        # =================================================

        model = (
            LandsatTransferModel()
            .to(DEVICE)
        )

        # =================================================
        # load pretrained backbone
        # =================================================

        load_uav_shared_weights(model)

        # =================================================
        # Stage1
        # =================================================

        print("\nStage1: train adapters")

        for name, p in model.named_parameters():

            if (
                "input_embed" not in name
                and "film" not in name
                and "head" not in name
            ):
                p.requires_grad = False

        optimizer = torch.optim.Adam(

            filter(
                lambda p: p.requires_grad,
                model.parameters()
            ),

            lr=1e-3
        )

        criterion = nn.MSELoss()

        # -------------------------------------------------
        # stage1 training
        # -------------------------------------------------

        for epoch in range(50):

            model.train()

            total_loss = 0

            for batch in train_loader:

                pred = model(

                    batch["x"].to(DEVICE),

                    batch["lambda"].to(DEVICE),

                    batch["delta"].to(DEVICE),

                    batch["type"].to(DEVICE),

                    batch["temp"].to(DEVICE)
                )

                loss = criterion(
                    pred,
                    batch["y"].to(DEVICE)
                )

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()

                total_loss += loss.item()

            mse, rmse, mae, r2 = evaluate(
                model,
                test_loader
            )

            print(
                f"[Fold {fold_id}] "
                f"[Stage1] "
                f"Epoch {epoch:03d} | "
                f"Loss {total_loss:.4f} | "
                f"R2 {r2:.4f}"
            )

        # =================================================
        # Stage2
        # =================================================

        print("\nStage2: finetune all")

        for p in model.parameters():

            p.requires_grad = True

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=1e-4
        )

        # =================================================
        # Early stopping
        # =================================================

        best_r2 = -999

        best_model = None

        patience = 50

        wait = 0

        # =================================================
        # finetune
        # =================================================

        for epoch in range(500):

            model.train()

            total_loss = 0

            for batch in train_loader:

                pred = model(

                    batch["x"].to(DEVICE),

                    batch["lambda"].to(DEVICE),

                    batch["delta"].to(DEVICE),

                    batch["type"].to(DEVICE),

                    batch["temp"].to(DEVICE)
                )

                loss = criterion(
                    pred,
                    batch["y"].to(DEVICE)
                )

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()

                total_loss += loss.item()

            # ---------------------------------------------
            # evaluate on TEST fold
            # ---------------------------------------------

            mse, rmse, mae, r2 = evaluate(
                model,
                test_loader
            )

            print(
                f"[Fold {fold_id}] "
                f"[Stage2] "
                f"Epoch {epoch:03d} | "
                f"Loss {total_loss:.4f} | "
                f"RMSE {rmse:.4f} | "
                f"R2 {r2:.4f}"
            )

            # ---------------------------------------------
            # save best
            # ---------------------------------------------

            if r2 > best_r2:

                best_r2 = r2

                best_model = copy.deepcopy(
                    model.state_dict()
                )

                wait = 0

                torch.save(

                    best_model,

                    os.path.join(
                        SAVE_DIR,
                        f"best_fold{fold_id}.pth"
                    )
                )

            else:

                wait += 1

                # -----------------------------------------
                # early stop
                # -----------------------------------------

                if wait >= patience:

                    print(
                        f"\nFold {fold_id} "
                        f"Early Stopping"
                    )

                    break

        # =================================================
        # final evaluation
        # =================================================

        model.load_state_dict(
            best_model
        )

        mse, rmse, mae, r2 = evaluate(
            model,
            test_loader
        )

        print("\n========================")
        print(f"FOLD {fold_id} RESULT")
        print("========================")

        print(f"MSE  : {mse:.4f}")
        print(f"RMSE : {rmse:.4f}")
        print(f"MAE  : {mae:.4f}")
        print(f"R2   : {r2:.4f}")

        all_results.append([

            mse,
            rmse,
            mae,
            r2
        ])

    # =====================================================
    # final CV result
    # =====================================================

    all_results = np.array(
        all_results
    )

    mean_result = all_results.mean(axis=0)

    std_result = all_results.std(axis=0)

    print("\n========================")
    print("5-FOLD FINAL RESULT")
    print("========================")

    metrics = [

        "MSE",
        "RMSE",
        "MAE",
        "R2"
    ]

    for i, m in enumerate(metrics):

        print(
            f"{m}: "
            f"{mean_result[i]:.4f} "
            f"± "
            f"{std_result[i]:.4f}"
        )

# =========================================================

if __name__ == "__main__":

    train()


LANDSAT-8 TRANSFER

================ FOLD 0 ================

Loading UAV shared backbone...
Loaded 29 shared parameters

Stage1: train adapters


C:\Users\PC\AppData\Local\Temp\ipykernel_47308\2308616086.py:584: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load(


[Fold 0] [Stage1] Epoch 000 | Loss 863.9841 | R2 -12.6957
[Fold 0] [Stage1] Epoch 001 | Loss 502.5305 | R2 -4.9293
[Fold 0] [Stage1] Epoch 002 | Loss 175.1627 | R2 -0.2918
[Fold 0] [Stage1] Epoch 003 | Loss 49.8111 | R2 0.2019
[Fold 0] [Stage1] Epoch 004 | Loss 46.3942 | R2 0.1124
[Fold 0] [Stage1] Epoch 005 | Loss 40.2814 | R2 0.2248
[Fold 0] [Stage1] Epoch 006 | Loss 41.4794 | R2 0.1851
[Fold 0] [Stage1] Epoch 007 | Loss 41.4116 | R2 0.1529
[Fold 0] [Stage1] Epoch 008 | Loss 39.6052 | R2 0.0702
[Fold 0] [Stage1] Epoch 009 | Loss 40.5571 | R2 0.1329
[Fold 0] [Stage1] Epoch 010 | Loss 42.6180 | R2 0.2210
[Fold 0] [Stage1] Epoch 011 | Loss 41.2906 | R2 0.2171
[Fold 0] [Stage1] Epoch 012 | Loss 41.1921 | R2 0.1592
[Fold 0] [Stage1] Epoch 013 | Loss 39.6271 | R2 0.2527
[Fold 0] [Stage1] Epoch 014 | Loss 40.3863 | R2 0.2206
[Fold 0] [Stage1] Epoch 015 | Loss 40.4805 | R2 0.0231
[Fold 0] [Stage1] Epoch 016 | Loss 39.1989 | R2 0.2294
[Fold 0] [Stage1] Epoch 017 | Loss 41.4816 | R2 0.1063
[Fo

C:\Users\PC\AppData\Local\Temp\ipykernel_47308\2308616086.py:584: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load(


[Fold 1] [Stage1] Epoch 002 | Loss 57.2199 | R2 -0.0676
[Fold 1] [Stage1] Epoch 003 | Loss 40.4989 | R2 0.0791
[Fold 1] [Stage1] Epoch 004 | Loss 35.4315 | R2 0.1875
[Fold 1] [Stage1] Epoch 005 | Loss 35.9907 | R2 0.1654
[Fold 1] [Stage1] Epoch 006 | Loss 35.3262 | R2 0.1369
[Fold 1] [Stage1] Epoch 007 | Loss 35.8161 | R2 0.1569
[Fold 1] [Stage1] Epoch 008 | Loss 35.2684 | R2 0.1828
[Fold 1] [Stage1] Epoch 009 | Loss 34.6248 | R2 0.1704
[Fold 1] [Stage1] Epoch 010 | Loss 34.4705 | R2 0.1763
[Fold 1] [Stage1] Epoch 011 | Loss 34.5042 | R2 0.1936
[Fold 1] [Stage1] Epoch 012 | Loss 34.7908 | R2 0.1589
[Fold 1] [Stage1] Epoch 013 | Loss 34.9386 | R2 0.1762
[Fold 1] [Stage1] Epoch 014 | Loss 34.1343 | R2 0.1903
[Fold 1] [Stage1] Epoch 015 | Loss 34.5198 | R2 0.1115
[Fold 1] [Stage1] Epoch 016 | Loss 34.8042 | R2 0.1861
[Fold 1] [Stage1] Epoch 017 | Loss 33.9023 | R2 0.1898
[Fold 1] [Stage1] Epoch 018 | Loss 33.7520 | R2 0.1349
[Fold 1] [Stage1] Epoch 019 | Loss 35.3460 | R2 0.1774
[Fold 1] 

...

[Fold 2] [Stage1] Epoch 001 | Loss 307.6668 | R2 -1.3337
[Fold 2] [Stage1] Epoch 002 | Loss 68.9511 | R2 -0.0243
[Fold 2] [Stage1] Epoch 003 | Loss 43.0050 | R2 0.2485
[Fold 2] [Stage1] Epoch 004 | Loss 45.5818 | R2 0.2982
[Fold 2] [Stage1] Epoch 005 | Loss 42.2609 | R2 0.1757
[Fold 2] [Stage1] Epoch 006 | Loss 43.9900 | R2 0.2169
[Fold 2] [Stage1] Epoch 007 | Loss 42.2741 | R2 0.2635
[Fold 2] [Stage1] Epoch 008 | Loss 39.9259 | R2 0.2301
[Fold 2] [Stage1] Epoch 009 | Loss 40.4675 | R2 0.2858
[Fold 2] [Stage1] Epoch 010 | Loss 42.6760 | R2 0.3040
[Fold 2] [Stage1] Epoch 011 | Loss 41.1828 | R2 0.2546
[Fold 2] [Stage1] Epoch 012 | Loss 40.5114 | R2 0.2214
[Fold 2] [Stage1] Epoch 013 | Loss 41.6037 | R2 0.1081
[Fold 2] [Stage1] Epoch 014 | Loss 40.6341 | R2 0.2400
[Fold 2] [Stage1] Epoch 015 | Loss 40.6507 | R2 0.2269
[Fold 2] [Stage1] Epoch 016 | Loss 41.0251 | R2 0.2352
[Fold 2] [Stage1] Epoch 017 | Loss 39.2538 | R2 0.1681
[Fold 2] [Stage1] Epoch 018 | Loss 41.4894 | R2 0.2267
[Fold 2

C:\Users\PC\AppData\Local\Temp\ipykernel_47308\2308616086.py:584: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load(


[Fold 3] [Stage1] Epoch 000 | Loss 749.5115 | R2 -9.7210
[Fold 3] [Stage1] Epoch 001 | Loss 361.5684 | R2 -2.5072
[Fold 3] [Stage1] Epoch 002 | Loss 99.4944 | R2 0.0155
[Fold 3] [Stage1] Epoch 003 | Loss 48.5185 | R2 0.0789
[Fold 3] [Stage1] Epoch 004 | Loss 45.0042 | R2 0.1701
[Fold 3] [Stage1] Epoch 005 | Loss 41.9354 | R2 0.2175
[Fold 3] [Stage1] Epoch 006 | Loss 38.7053 | R2 0.2075
[Fold 3] [Stage1] Epoch 007 | Loss 41.1284 | R2 0.1939
[Fold 3] [Stage1] Epoch 008 | Loss 40.0005 | R2 0.2329
[Fold 3] [Stage1] Epoch 009 | Loss 39.3709 | R2 0.2408
[Fold 3] [Stage1] Epoch 010 | Loss 38.8491 | R2 0.2294
[Fold 3] [Stage1] Epoch 011 | Loss 37.2965 | R2 0.2308
[Fold 3] [Stage1] Epoch 012 | Loss 38.3104 | R2 0.2262
[Fold 3] [Stage1] Epoch 013 | Loss 38.0701 | R2 0.2280
[Fold 3] [Stage1] Epoch 014 | Loss 38.3848 | R2 0.2036
[Fold 3] [Stage1] Epoch 015 | Loss 39.2294 | R2 0.2104
[Fold 3] [Stage1] Epoch 016 | Loss 39.1142 | R2 0.2272
[Fold 3] [Stage1] Epoch 017 | Loss 37.4920 | R2 0.2401
[Fold 

C:\Users\PC\AppData\Local\Temp\ipykernel_47308\2308616086.py:584: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load(


[Fold 4] [Stage1] Epoch 000 | Loss 785.2169 | R2 -11.0167
[Fold 4] [Stage1] Epoch 001 | Loss 379.1178 | R2 -2.6091
[Fold 4] [Stage1] Epoch 002 | Loss 96.7951 | R2 -0.0363
[Fold 4] [Stage1] Epoch 003 | Loss 54.0285 | R2 -0.1029
[Fold 4] [Stage1] Epoch 004 | Loss 47.9913 | R2 -0.0951
[Fold 4] [Stage1] Epoch 005 | Loss 49.0343 | R2 0.0458
[Fold 4] [Stage1] Epoch 006 | Loss 46.0473 | R2 0.1913
[Fold 4] [Stage1] Epoch 007 | Loss 44.8041 | R2 0.0057
[Fold 4] [Stage1] Epoch 008 | Loss 44.7485 | R2 0.1660
[Fold 4] [Stage1] Epoch 009 | Loss 43.9399 | R2 0.1689
[Fold 4] [Stage1] Epoch 010 | Loss 42.4569 | R2 0.1685
[Fold 4] [Stage1] Epoch 011 | Loss 42.3744 | R2 0.1610
[Fold 4] [Stage1] Epoch 012 | Loss 41.8134 | R2 0.2243
[Fold 4] [Stage1] Epoch 013 | Loss 40.9707 | R2 0.2250
[Fold 4] [Stage1] Epoch 014 | Loss 38.9701 | R2 0.2652
[Fold 4] [Stage1] Epoch 015 | Loss 39.9594 | R2 0.2529
[Fold 4] [Stage1] Epoch 016 | Loss 39.1357 | R2 0.2259
[Fold 4] [Stage1] Epoch 017 | Loss 40.7861 | R2 0.2306
[F

# muti task learning

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import copy
import torch.nn.functional as F
from collections import defaultdict

DATA_DIR = r"\data\input"
model_save_dir = os.path.join(DATA_DIR, "MTL-CSF-FPE-V3")
os.makedirs(model_save_dir, exist_ok=True)

# =========================
# ⭐ MTL dataset config
# =========================
selected_datasets = ["uav", "sentinel", "landsat", "uav2s2", "uav2l8"]

DATASET_CONFIG = {
    "uav": (0, "UAV.csv", "UAV/uav_splits.json"),
    "sentinel": (1, "Sentinel-2.csv", "Sentinel-2/sentinel_splits.json"),
    "landsat": (2, "Landsat-8.csv", "Landsat-8/landsat_splits.json"),
    "uav2s2": (3, "UAV_Sentinel_S2A.csv", "UAV-Sentinel2S2A/uav2s2_splits.json"),
    "uav2l8": (4, "UAV_Landsat8_SRF.csv", "UAV-Landsat8SRF/uav2l8_splits.json")
}

# =========================
# 0. wavelength normalization
# =========================
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002

def normalize_wavelength(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

def normalize_delta(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

# =========================
# Dataset
# =========================
class SpectralDataset(Dataset):
    def __init__(self, file_path, dtype, task_id, x_mean, x_std):
        """
        dtype: 数据类型编号 (0=UAV, 1=Sentinel-2, 2=Landsat-8, 3=UAV2S2, 4=UAV2L8)
        task_id: 任务编号（用于多任务学习）
        x_mean, x_std: 该任务的归一化参数（已单独计算）
        """
        df = pd.read_csv(file_path, encoding='gbk')
        self.dtype = dtype
        self.task_id = task_id

        temp = df.iloc[:, -2].values.astype(np.float32)
        self.x = df.iloc[:, :-2].values.astype(np.float32)
        self.y = df.iloc[:, -1].values.astype(np.float32)

        # NDWI filter
        if dtype == 1:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 2:
            ndwi = (df['SR_B3'] - df['SR_B5']) / (df['SR_B3'] + df['SR_B5'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # clean
        mask = np.all(self.x >= 0, axis=1)
        self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        self.x = (self.x - x_mean) / (x_std + 1e-6)

        # wavelength
        if dtype == 0:
            self.lambda_ = np.arange(350, 1003, 4).astype(np.float32)
            self.delta = np.full_like(self.lambda_, 4.0)

        elif dtype == 1 or dtype == 3:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),
                     (740,15),(783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        elif dtype == 2 or dtype == 4:
            bands = [(443,20),(482,65),(561,75),(655,50),(865,40)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        self.lambda_ = normalize_wavelength(self.lambda_)
        self.delta = normalize_delta(self.delta)

        self.temp = (temp - temp.mean()) / (temp.std() + 1e-6)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(self.x[idx]),
            "lambda": torch.tensor(self.lambda_),
            "delta": torch.tensor(self.delta),
            "y": torch.tensor(self.y[idx]),
            "dtype": torch.tensor(self.dtype),
            "task_id": torch.tensor(self.task_id),
            "temp": torch.tensor(self.temp[idx])
        }

# =========================
# CSF: Common Spectral Fusion (简化版V3)
# =========================
class CommonSpectralFusion(nn.Module):
    """
    Shared Spectral Fusion Module (Simplified Design)

    This module implements a unified spectral representation learning pipeline
    shared across different remote sensing modalities.

    Pipeline:

    1. Input Construction:
    Concatenate spectral value, wavelength, and bandwidth:
    [x, λ, δ] → (B, N, 3)

    2. Domain Conditioning:
    Encode sensor type (dtype) as a global conditioning vector
    applied via residual modulation.

    3. Feature Embedding:
    Project input into a unified latent space:
    (B, N, 3) → (B, N, d_model)

    4. Spectral Projection:
    Apply Gaussian kernel projection onto a fixed spectral grid:
    (B, N, d_model) → (B, G, d_model)

    5. Positional Encoding & Fusion:
    Inject Fourier-based positional encoding and fuse representations:
    (B, G, d_model) → (B, G, d_model)

    Note:
    - All modalities share the same embedding, projection, and fusion parameters.
    - Only sensor-type conditioning introduces domain-specific variation.
    """
    def __init__(self, d_model=64, grid_size=64, num_tasks=5):
        super().__init__()
        self.d_model = d_model

        # fixed spectral grid
        self.register_buffer("grid", torch.linspace(0, 1, grid_size))

        # Fourier position encoding
        self.fourier_B = nn.Parameter(torch.logspace(0, 2, d_model // 2))
        self.fourier_scale = nn.Parameter(torch.ones(1))

        # fusion layer
        self.fusion = nn.Linear(d_model * 2, d_model)

        # =========================
        # Domain Alignment: dtype embedding
        # =========================
        self.dtype_embed = nn.Embedding(num_tasks, d_model)

        # input embedding: shared across all tasks
        self.input_embed = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

    def fourier_encoding(self, x):
        """Fourier positional encoding for spectral grid"""
        x = x.unsqueeze(-1)  # (B, N, 1)
        proj = x * self.fourier_B * self.fourier_scale
        return torch.cat([
            torch.sin(proj),
            torch.cos(proj)
        ], dim=-1)

    def spectral_project(self, z, wl, delta):
        """Spectral projection onto a fixed grid"""
        grid = self.grid.to(z.device).unsqueeze(0).unsqueeze(0)

        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        sigma = delta + 1e-6
        weight = torch.exp(-((wl - grid) ** 2) / (sigma ** 2))
        weight = weight / (weight.sum(dim=1, keepdim=True) + 1e-8)

        return (z.unsqueeze(2) * weight.unsqueeze(-1)).sum(dim=1)

    def forward(self, x, wl, delta, dtype):
        """
        x: (B, C) - reflectance values
        wl: (B, C) - wavelengths
        delta: (B, C) - wavelength intervals
        dtype: (B,) - data type ID
        """
        # -------------------------
        # 1. input concat
        # -------------------------
        x = x.unsqueeze(-1)
        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        inp = torch.cat([x, wl, delta], dim=-1)

        # -------------------------
        # 2. domain alignment (ONLY dtype)
        # -------------------------
        dtype_emb = self.dtype_embed(dtype).unsqueeze(1)   # (B, 1, d)

        # -------------------------
        # 3. shared input embedding
        # -------------------------
        z = self.input_embed(inp)

        # fuse dtype (simple residual conditioning)
        z = z + dtype_emb

        # -------------------------
        # 4. shared spectral projection (CSF)
        # -------------------------
        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1)
        )

        # -------------------------
        # 5. shared Fourier positional encoding (FPE)
        # -------------------------
        grid = self.grid.unsqueeze(0).expand(z.size(0), -1)
        pos = self.fourier_encoding(grid)

        z = torch.cat([z, pos], dim=-1)
        z = self.fusion(z)

        return z

# =========================
# FPE: Feature Processing Engine
# =========================
class TaskSpecificHead(nn.Module):
    """task-specific head for regression, shared across tasks but conditioned on temperature"""
    def __init__(self, d_model=64):
        super().__init__()
        self.d_model = d_model

        # temperature embedding
        self.temp_embed = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # transformer encoder (shared)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # regression head
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, z, temp):
        """
        z: (B, grid_size, d_model) - features from CSF
        temp: (B,) - temperature
        """
        B = z.size(0)

        # -------------------------
        # 6. shared global token (temp + CLS)
        # -------------------------
        t = self.temp_embed(temp.unsqueeze(-1)).unsqueeze(1)   # (B, 1, d)

        z = torch.cat([t, z], dim=1)

        # -------------------------
        # 7. shared Transformer encoder
        # -------------------------
        z = self.encoder(z)

        z = z[:, 0]

        # -------------------------
        # 8. task-specific head
        # -------------------------
        return self.head(z).squeeze()

# =========================
# MTL:multi-task learning model
# =========================
class MTLSpectralModel(nn.Module):
    """multi-task learning model - simplified version V3"""
    def __init__(self, d_model=64, grid_size=64, num_tasks=5):
        super().__init__()
        self.csf = CommonSpectralFusion(d_model, grid_size, num_tasks)
        self.fpe = TaskSpecificHead(d_model)

    def forward(self, x, wl, delta, dtype, temp):
        """
        x: (B, C) - reflectance values
        wl: (B, C) - wavelengths
        delta: (B, C) - wavelength intervals
        dtype: (B,) - 数据类型 ID
        temp: (B,) - 温度
        """
        z = self.csf(x, wl, delta, dtype)
        pred = self.fpe(z, temp)
        return pred

# =========================
# Loss
# =========================
def r2_aware_loss(pred, target):
    """R2-aware loss function"""
    mse = torch.mean((pred - target) ** 2)
    pred_c = pred - pred.mean()
    tgt_c = target - target.mean()
    corr = torch.sum(pred_c * tgt_c) / (
        torch.sqrt(torch.sum(pred_c ** 2)) *
        torch.sqrt(torch.sum(tgt_c ** 2)) + 1e-8
    )
    return mse + 0.5 * (1 - corr)

# =========================
# Evaluation
# =========================
def evaluate_mtl(model, dataloaders, device):
    """multi-task evaluation"""
    model.eval()
    results = defaultdict(dict)

    with torch.no_grad():
        for task_name, loader in dataloaders.items():
            y_true, y_pred = [], []

            for batch in loader:
                pred = model(
                    batch["x"].to(device),
                    batch["lambda"].to(device),
                    batch["delta"].to(device),
                    batch["dtype"].to(device),
                    batch["temp"].to(device)
                )

                y_true.extend(batch["y"].cpu().numpy())
                y_pred.extend(pred.cpu().numpy())

            y_true = np.array(y_true)
            y_pred = np.array(y_pred)

            mse = mean_squared_error(y_true, y_pred)
            rmse = np.sqrt(mse)
            mae = mean_absolute_error(y_true, y_pred)
            r2 = r2_score(y_true, y_pred)

            results[task_name] = {"mse": mse, "rmse": rmse, "mae": mae, "r2": r2}

    return results

# =========================
# Training
# =========================
def train_mtl():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    print("\n" + "="*70)
    print("Step 1: Load normalization parameters for each task (computed separately, not merged)")
    print("="*70)
    
    norm_params = {}
    for name in selected_datasets:
        dtype, csv_file, _ = DATASET_CONFIG[name]
        df = pd.read_csv(os.path.join(DATA_DIR, csv_file), encoding='gbk')
        x = df.iloc[:, :-2].values.astype(np.float32)

        x_mean = x.mean(axis=0)
        x_std = x.std(axis=0) + 1e-6
        norm_params[name] = (x_mean, x_std)
        
        print(f"  {name:12s}: x_shape={x.shape}, mean_shape={x_mean.shape}")

    # load datasets and splits
    print("\n" + "="*70)
    print("Step 2: Load datasets and split strategies")
    print("="*70)
    
    datasets = {}
    splits = {}

    for task_id, name in enumerate(selected_datasets):
        dtype, csv_file, split_file = DATASET_CONFIG[name]
        csv_path = os.path.join(DATA_DIR, csv_file)
        split_path = os.path.join(DATA_DIR, split_file)

        x_mean, x_std = norm_params[name]
        dataset = SpectralDataset(csv_path, dtype, task_id, x_mean, x_std)
        datasets[name] = dataset

        try:
            with open(split_path, 'r') as f:
                folds = json.load(f)
            splits[name] = folds
        except FileNotFoundError:
            print(f"  Creating random splits for {name}")
            n = len(dataset)
            fold_splits = []
            for _ in range(5):
                idx = np.random.permutation(n)
                fold_splits.append({
                    "train_idx": idx[:int(0.8*n)].tolist(),
                    "test_idx": idx[int(0.8*n):].tolist()
                })
            splits[name] = fold_splits

    print(f"  Loaded {len(datasets)} datasets")

    # 5-fold 交叉验证
    print("\n" + "="*70)
    print("Step 3: 5-fold cross-validation training")
    print("="*70)
    
    fold_results = defaultdict(list)

    for fold_id in range(5):
        print(f"\n{'='*60}")
        print(f"FOLD {fold_id}")
        print(f"{'='*60}")

        train_loaders = {}
        test_loaders = {}

        for name in selected_datasets:
            dataset = datasets[name]
            train_idx = np.array(splits[name][fold_id]["train_idx"])
            test_idx = np.array(splits[name][fold_id]["test_idx"])

            train_loaders[name] = DataLoader(Subset(dataset, train_idx), batch_size=16, shuffle=True)
            test_loaders[name] = DataLoader(Subset(dataset, test_idx), batch_size=16)

        # model
        model = MTLSpectralModel(d_model=64, grid_size=64, num_tasks=len(selected_datasets)).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=3e-4)

        best_total_rmse = float("inf")
        patience = 50
        wait = 0
        best_model = None

        for epoch in range(500):
            model.train()
            total_loss = 0
            num_batches = 0

            for task_name in selected_datasets:
                for batch in train_loaders[task_name]:
                    pred = model(
                        batch["x"].to(device),
                        batch["lambda"].to(device),
                        batch["delta"].to(device),
                        batch["dtype"].to(device),
                        batch["temp"].to(device)
                    )

                    loss = r2_aware_loss(pred, batch["y"].to(device))
                    opt.zero_grad()
                    loss.backward()
                    opt.step()

                    total_loss += loss.item()
                    num_batches += 1

            if epoch % 5 == 0:
                eval_results = evaluate_mtl(model, test_loaders, device)
                avg_rmse = np.mean([eval_results[name]["rmse"] for name in selected_datasets])

                print(f"Epoch {epoch:03d} | Loss {total_loss/num_batches:.4f} | Avg RMSE {avg_rmse:.4f}")

                if avg_rmse < best_total_rmse:
                    best_total_rmse = avg_rmse
                    best_model = copy.deepcopy(model.state_dict())
                    wait = 0
                    torch.save(best_model, os.path.join(model_save_dir, f"best_model_fold{fold_id}.pth"))
                else:
                    wait += 1
                    if wait >= patience:
                        print("Early stopping")
                        break

        # model evaluation
        print(f"\n[FOLD {fold_id} FINAL]")
        model.load_state_dict(best_model)
        final_results = evaluate_mtl(model, test_loaders, device)

        for task_name in selected_datasets:
            r = final_results[task_name]
            print(f"{task_name:12s}: MSE={r['mse']:.4f} RMSE={r['rmse']:.4f} MAE={r['mae']:.4f} R2={r['r2']:.4f}")
            fold_results[task_name].append(r)

    # overall 5-fold results
    print(f"\n{'='*70}")
    print("5-FOLD RESULTS")
    print(f"{'='*70}")

    for task_name in selected_datasets:
        results = fold_results[task_name]
        print(f"\n{task_name}:")
        print(f"  MSE:  {np.mean([r['mse'] for r in results]):.4f} ± {np.std([r['mse'] for r in results]):.4f}")
        print(f"  RMSE: {np.mean([r['rmse'] for r in results]):.4f} ± {np.std([r['rmse'] for r in results]):.4f}")
        print(f"  MAE:  {np.mean([r['mae'] for r in results]):.4f} ± {np.std([r['mae'] for r in results]):.4f}")
        print(f"  R2:   {np.mean([r['r2'] for r in results]):.4f} ± {np.std([r['r2'] for r in results]):.4f}")

if __name__ == "__main__":
    train_mtl()

Using device: cuda

第1步：加载各任务的统计参数（单独计算，不合并）
  uav         : x_shape=(570, 164), mean_shape=(164,)
  sentinel    : x_shape=(1764, 10), mean_shape=(10,)
  landsat     : x_shape=(640, 5), mean_shape=(5,)
  uav2s2      : x_shape=(570, 10), mean_shape=(10,)
  uav2l8      : x_shape=(570, 5), mean_shape=(5,)

第2步：加载数据集和分割策略
  Creating random splits for sentinel
  Creating random splits for landsat
  Loaded 5 datasets

第3步：5-折交叉验证训练

FOLD 0
Epoch 000 | Loss 12.0395 | Avg RMSE 1.6745
Epoch 005 | Loss 2.2839 | Avg RMSE 1.4725
Epoch 010 | Loss 2.1202 | Avg RMSE 1.4648
Epoch 015 | Loss 2.0463 | Avg RMSE 1.5511
Epoch 020 | Loss 2.0095 | Avg RMSE 1.4903
Epoch 025 | Loss 1.9802 | Avg RMSE 1.3372
Epoch 030 | Loss 1.8149 | Avg RMSE 1.1141
Epoch 035 | Loss 1.7402 | Avg RMSE 1.2889
Epoch 040 | Loss 1.6777 | Avg RMSE 1.1547
Epoch 045 | Loss 1.6311 | Avg RMSE 1.2928
Epoch 050 | Loss 1.6073 | Avg RMSE 1.3330
Epoch 055 | Loss 1.6292 | Avg RMSE 1.1955
Epoch 060 | Loss 1.5416 | Avg RMSE 1.1693
Epoch 065 | Los